# R3: independent attackers, budget, knowledge, malicious share, architecture, 10 more seeds (docs/PREREG_R3.md)
Pre-registered in docs/PREREG_REPLICATION.md (committed before launch). The scaler is fitted on tenant training rows only. NEW seeds for every study. Every stage is guarded: a failure prints and the notebook continues. *Accelerator = GPU T4 x2, Internet = On; add the Kubernetes dataset (redamorsli/kube-ids0) as an input before a background run.*

In [ ]:
RUN = {'H1B': True, 'ES': True, 'AGR': True, 'FRAC': True, 'KNOW': True, 'BUDGET2': True, 'ARCH': True, 'BUDGET8': True}
print('\n✔ CELL COMPLETED: switches set')

In [ ]:
!pip -q install kagglehub pyarrow cryptography psutil
try:
    import subprocess, sys
    subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'liboqs-python'], check=True, timeout=1500)
except Exception as e:
    print('liboqs-python install failed; the post-quantum cell will be skipped:', e)
print('\n✔ CELL COMPLETED: packages installed')

In [ ]:
# Writes the CURRENT qbfl package next to this notebook (scaler fitted on tenant training rows only).
import pathlib, json
FILES = json.loads("{\"accountability.py\": \"\\\"\\\"\\\"Accountability layer: WHEN to slash a tenant, separate from how updates are aggregated.\\n\\nSlashing is irreversible and costs the tenant its bond, so it needs a different standard of evidence from\\ndown-weighting one update. Four rules are compared side by side in the same run (observational: they record\\nwho WOULD be slashed and when; they do not change aggregation):\\n\\n  margin   slash when one round's score exceeds the defense's pass mark by a fixed margin (the naive rule)\\n  kofm     slash after k offences within the last m rounds (common heuristic)\\n  peer_ctm anytime-valid e-process on conformal p-values of the tenant's score AMONG ITS PEERS in each round\\n  self_ctm anytime-valid e-process on conformal p-values of the tenant's peer-rank AGAINST ITS OWN PAST ranks\\n           (\\\"slash on change, not on difference\\\")\\n  peer_bet / self_bet  same p-values, bounded betting e-process  E_t = prod (1 + lam (1 - 2 p_s)),  lam = 0.5.\\n           Also a test supermartingale for super-uniform p (same Ville bound), but each factor is in [0.5, 1.5], so a slightly\\n           mis-calibrated p-value cannot blow the evidence up; it needs about 7-8 strongly suspicious rounds to reach 1/alpha = 20.\\n\\nFor the two *_ctm rules the running evidence is E_t = prod_s f(p_s) with the calibrator f(p) = k p^(k-1), 0<k<1.\\nIf, for an honest tenant, each p_s is (conditionally) super-uniform, E_t is a test supermartingale and Ville's\\ninequality gives  P(sup_t E_t >= 1/alpha) <= alpha : an honest tenant is EVER slashed with probability at most alpha.\\npeer_ctm needs honest tenants to be exchangeable with each other (fails under non-IID data);\\nself_ctm needs an honest tenant's own peer-rank sequence to be exchangeable over time (stationary behaviour).\\nWhether these hold on real data is measured, not assumed.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport torch\\n\\n\\ndef badness_scores(U: torch.Tensor, defense) -> np.ndarray:\\n    \\\"\\\"\\\"Per-tenant badness this round (larger = more suspicious). Uses the defense's own score when it has one,\\n    otherwise a defense-agnostic one (distance to the coordinate-wise median, scaled by its median).\\\"\\\"\\\"\\n    last = getattr(defense, \\\"last\\\", None) or {}\\n    if \\\"dmg\\\" in last and len(last[\\\"dmg\\\"]) == U.shape[0]:\\n        return np.asarray(last[\\\"dmg\\\"], dtype=float)\\n    med = U.median(0).values\\n    d = (U - med).norm(dim=1)\\n    return (d / (d.median() + 1e-12)).detach().cpu().numpy().astype(float)\\n\\n\\nclass Accountability:\\n    ALPHAS = (0.01, 0.05, 0.1)\\n\\n    def __init__(self, n: int, alpha: float = 0.05, kappa_cal: float = 0.5, warmup: int = 5, k: int = 3, m: int = 5,\\n                 margin_mult: float = 1.5, min_hist: int = 5, seed: int = 0):\\n        self.n, self.alpha, self.kc, self.warmup = n, alpha, kappa_cal, warmup\\n        self.k, self.m, self.margin_mult, self.min_hist = k, m, margin_mult, min_hist\\n        self.rng = np.random.default_rng(seed + 991)\\n        self.logE = {\\\"peer_ctm\\\": np.zeros(n), \\\"self_ctm\\\": np.zeros(n), \\\"peer_bet\\\": np.zeros(n), \\\"self_bet\\\": np.zeros(n), \\\"raw_ctm\\\": np.zeros(n), \\\"raw_bet\\\": np.zeros(n)}\\n        self.lam = 0.5   # bounded bet: factor 1 + lam (1 - 2p) lies in [1 - lam, 1 + lam]\\n        self.rank_hist = [[] for _ in range(n)]\\n        self.offences = [[] for _ in range(n)]\\n        self.E_RULES = (\\\"peer_ctm\\\", \\\"self_ctm\\\", \\\"peer_bet\\\", \\\"self_bet\\\", \\\"raw_ctm\\\", \\\"raw_bet\\\")\\n        self.slashed_at = {r: np.full(n, -1) for r in (\\\"margin\\\", \\\"kofm\\\")}\\n        for r_ in self.E_RULES:\\n            for a in self.ALPHAS:\\n                self.slashed_at[f\\\"{r_}@{a}\\\"] = np.full(n, -1)\\n        self.raw_hist = [[] for _ in range(n)]\\n        self.thr = np.log(1.0 / alpha)\\n        self.trace = None            # set to [] to record per-round scores and p-values (diagnostics)\\n\\n    def _calib(self, p: np.ndarray) -> np.ndarray:\\n        p = np.clip(p, 1e-12, 1.0)\\n        return np.log(self.kc) + (self.kc - 1.0) * np.log(p)\\n\\n    def _conformal_peer(self, s: np.ndarray) -> np.ndarray:\\n        \\\"\\\"\\\"p_i = (#{j: s_j > s_i} + U * #{j: s_j == s_i}) / n  (smoothed, randomised; small p = unusually bad).\\\"\\\"\\\"\\n        n = len(s); p = np.empty(n)\\n        for i in range(n):\\n            gt = np.sum(s > s[i]); eq = np.sum(s == s[i])\\n            p[i] = (gt + self.rng.random() * eq) / n\\n        return p\\n\\n    def update(self, r: int, U: torch.Tensor, defense):\\n        s = badness_scores(U, defense)\\n        n = len(s)\\n        rank = (np.argsort(np.argsort(s)) + 0.5) / n                   # 0 = best, 1 = worst this round\\n        last = getattr(defense, \\\"last\\\", None) or {}\\n        thr = last.get(\\\"thr\\\", None)\\n        offence = (s > (thr * self.margin_mult if thr not in (None, float(\\\"inf\\\")) and np.isfinite(thr) else np.quantile(s, 0.9) * self.margin_mult))\\n        if r < self.warmup:                                              # no slashing while the model is still forming\\n            for i in range(n):\\n                self.rank_hist[i].append(rank[i]); self.raw_hist[i].append(s[i])\\n            return\\n        # own ABSOLUTE score vs own past (no peer ranks, so other tenants cannot shift it)\\n        p_raw = np.ones(n)\\n        for i in range(n):\\n            h = np.asarray(self.raw_hist[i])\\n            if len(h) >= self.min_hist:\\n                gt = np.sum(h > s[i]); eq = np.sum(h == s[i])\\n                p_raw[i] = (gt + self.rng.random() * (eq + 1)) / (len(h) + 1)\\n            self.raw_hist[i].append(s[i])\\n        self.logE[\\\"raw_ctm\\\"] += self._calib(p_raw)\\n        self.logE[\\\"raw_bet\\\"] += np.log1p(self.lam * (1 - 2 * p_raw))\\n        # naive per-round margin\\n        for i in np.flatnonzero(offence):\\n            if self.slashed_at[\\\"margin\\\"][i] < 0:\\n                self.slashed_at[\\\"margin\\\"][i] = r\\n        # k offences in the last m rounds\\n        for i in range(n):\\n            self.offences[i].append(bool(offence[i]))\\n            if self.slashed_at[\\\"kofm\\\"][i] < 0 and sum(self.offences[i][-self.m:]) >= self.k:\\n                self.slashed_at[\\\"kofm\\\"][i] = r\\n        # peer conformal e-process\\n        p_peer = self._conformal_peer(s)\\n        self.logE[\\\"peer_ctm\\\"] += self._calib(p_peer)\\n        # self-referential conformal e-process on the tenant's own rank history\\n        p_self = np.ones(n)\\n        for i in range(n):\\n            h = np.asarray(self.rank_hist[i])\\n            if len(h) >= self.min_hist:\\n                gt = np.sum(h > rank[i]); eq = np.sum(h == rank[i])\\n                p_self[i] = (gt + self.rng.random() * (eq + 1)) / (len(h) + 1)\\n            self.rank_hist[i].append(rank[i])\\n        self.logE[\\\"self_ctm\\\"] += self._calib(p_self)\\n        # bounded-bet e-processes: E[1 + lam (1 - 2p)] = 1 for uniform p, and each factor is in [0.5, 1.5]\\n        self.logE[\\\"peer_bet\\\"] += np.log1p(self.lam * (1 - 2 * p_peer))\\n        self.logE[\\\"self_bet\\\"] += np.log1p(self.lam * (1 - 2 * p_self))\\n        if self.trace is not None:\\n            self.trace.append(dict(r=int(r), s=np.round(s, 5).tolist(), p_peer=np.round(p_peer, 5).tolist(),\\n                                   p_self=np.round(p_self, 5).tolist(), p_raw=np.round(p_raw, 5).tolist()))\\n        for rule in self.E_RULES:\\n            for a in self.ALPHAS:\\n                key = f\\\"{rule}@{a}\\\"\\n                hit = (self.logE[rule] >= np.log(1.0 / a)) & (self.slashed_at[key] < 0)\\n                self.slashed_at[key][hit] = r\\n\\n    def summary(self, mal: list, attack_start: int) -> dict:\\n        mal = set(mal); hon = [i for i in range(self.n) if i not in mal]\\n        out = {}\\n        for rule, at in self.slashed_at.items():\\n            out[f\\\"acct_{rule}_honest_slashed\\\"] = float(np.mean([at[i] >= 0 for i in hon])) if hon else float(\\\"nan\\\")\\n            out[f\\\"acct_{rule}_mal_slashed\\\"] = float(np.mean([at[i] >= 0 for i in mal])) if mal else float(\\\"nan\\\")\\n            delays = [at[i] - attack_start for i in mal if at[i] >= 0]\\n            out[f\\\"acct_{rule}_rounds_to_slash\\\"] = float(np.mean(delays)) if delays else float(\\\"nan\\\")\\n        return out\\n\", \"analysis.py\": \"\\\"\\\"\\\"Turn the JSONL results into tables, paired tests and figures.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport glob\\nimport json\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n\\ndef load(pattern: str) -> pd.DataFrame:\\n    rows = []\\n    for f in glob.glob(pattern):\\n        for line in open(f):\\n            r = json.loads(line)\\n            if \\\"error\\\" in r:\\n                continue\\n            fin = r.pop(\\\"final\\\")\\n            r.pop(\\\"curve\\\", None)\\n            r.update({f\\\"final_{k}\\\": v for k, v in fin.items()})\\n            rows.append(r)\\n    df = pd.DataFrame(rows).drop_duplicates(\\\"run_id\\\")\\n    return df\\n\\n\\ndef with_drop(df: pd.DataFrame) -> pd.DataFrame:\\n    \\\"\\\"\\\"Attack effect = clean-run score minus attacked score, paired by defense and seed.\\\"\\\"\\\"\\n    base = df[df.attack == \\\"none\\\"].groupby([\\\"defense\\\", \\\"seed\\\"])[[\\\"final_macro_f1\\\", \\\"final_rec_target\\\"]].mean()\\n    base.columns = [\\\"clean_f1\\\", \\\"clean_rec_target\\\"]\\n    d = df.merge(base.reset_index(), on=[\\\"defense\\\", \\\"seed\\\"], how=\\\"left\\\")\\n    d[\\\"f1_drop\\\"] = d.clean_f1 - d.final_macro_f1\\n    d[\\\"target_recall_drop\\\"] = d.clean_rec_target - d.final_rec_target\\n    return d\\n\\n\\ndef summary(df: pd.DataFrame, mal_frac=0.3) -> pd.DataFrame:\\n    d = with_drop(df)\\n    d = d[(d.mal_frac == mal_frac) | (d.attack == \\\"none\\\")]\\n    g = d.groupby([\\\"attack\\\", \\\"defense\\\"]).agg(\\n        f1=(\\\"final_macro_f1\\\", \\\"mean\\\"), f1_sd=(\\\"final_macro_f1\\\", \\\"std\\\"),\\n        rec_target=(\\\"final_rec_target\\\", \\\"mean\\\"), rec_target_sd=(\\\"final_rec_target\\\", \\\"std\\\"),\\n        target_drop=(\\\"target_recall_drop\\\", \\\"mean\\\"),\\n        honest_rejected=(\\\"honest_rejected\\\", \\\"mean\\\"), mal_rejected=(\\\"mal_rejected\\\", \\\"mean\\\"),\\n        mass_on_mal=(\\\"mass_on_mal\\\", \\\"mean\\\"), n=(\\\"seed\\\", \\\"count\\\")).round(3)\\n    return g\\n\\n\\ndef paired_test(df: pd.DataFrame, attack: str, a: str, b: str, metric=\\\"final_rec_target\\\", mal_frac=0.3):\\n    \\\"\\\"\\\"Wilcoxon signed-rank across seeds: is defense `a` better than `b` under `attack`?\\\"\\\"\\\"\\n    from scipy.stats import wilcoxon\\n\\n    d = df[(df.attack == attack) & (df.mal_frac == mal_frac)]\\n    pa = d[d.defense == a].set_index(\\\"seed\\\")[metric]\\n    pb = d[d.defense == b].set_index(\\\"seed\\\")[metric]\\n    idx = pa.index.intersection(pb.index)\\n    if len(idx) < 3:\\n        return dict(n=len(idx), note=\\\"too few paired seeds\\\")\\n    diff = (pa[idx] - pb[idx]).values\\n    try:\\n        stat = wilcoxon(diff, alternative=\\\"greater\\\")\\n        p = float(stat.pvalue)\\n    except ValueError:\\n        p = float(\\\"nan\\\")\\n    return dict(n=len(idx), mean_diff=float(diff.mean()), p_value=p)\\n\\n\\ndef select_best(df: pd.DataFrame, max_honest_rejected: float = 0.10, mal_frac: float = 0.3):\\n    \\\"\\\"\\\"Pick one setting per defense from the TUNING runs.\\n\\n    Score = the worst DoS-family recall over {no attack, rule-aware, rule-aware with reputation farming}, so a setting\\n    cannot win by being strong under attack but poor on clean traffic. Settings that reject more than\\n    `max_honest_rejected` of honest tenants without an attack are not eligible (if none qualifies, the least\\n    harmful one is taken). Ties go to the higher clean macro-F1.\\\"\\\"\\\"\\n    d = df.copy()\\n    if \\\"dparams\\\" not in d:\\n        d[\\\"dparams\\\"] = \\\"{}\\\"\\n    rows = []\\n    for (defense, dp), g in d.groupby([\\\"defense\\\", \\\"dparams\\\"]):\\n        per = g.groupby(\\\"attack\\\")[\\\"final_rec_target\\\"].mean()\\n        if not {\\\"none\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\"} <= set(per.index):\\n            continue\\n        clean = g[g.attack == \\\"none\\\"]\\n        hr = clean[\\\"honest_rejected\\\"].fillna(0).mean()\\n        rows.append(dict(defense=defense, dparams=dp, worst_recall=per[[\\\"none\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\"]].min(),\\n                         clean_recall=per[\\\"none\\\"], rule_aware=per[\\\"rule_aware\\\"], farm=per[\\\"rule_aware_farm\\\"],\\n                         clean_f1=clean[\\\"final_macro_f1\\\"].mean(), honest_rejected=hr, n=len(g)))\\n    T = pd.DataFrame(rows)\\n    best = {}\\n    for defense, g in T.groupby(\\\"defense\\\"):\\n        ok = g[g.honest_rejected <= max_honest_rejected]\\n        pool = ok if len(ok) else g.sort_values(\\\"honest_rejected\\\").head(1)\\n        pick = pool.sort_values([\\\"worst_recall\\\", \\\"clean_f1\\\"], ascending=False).iloc[0]\\n        best[defense] = json.loads(pick.dparams)\\n    return best, T.sort_values([\\\"defense\\\", \\\"worst_recall\\\"], ascending=[True, False])\\n\", \"attacks.py\": \"\\\"\\\"\\\"Attackers.\\n\\nGoal of every attacker: make the global model miss a target attack family\\n(default \\\"dos\\\") while overall accuracy stays high. This is a stealthy\\nevasion goal; it is what a real intruder wants from a shared IDS.\\n\\nFixed attackers (do not read the contract):\\n    targeted_lf  label-flip data poisoning\\n    signflip     sign-flipped honest update\\n    lie          \\\"a little is enough\\\" (Baruch et al. 2019)\\n    minmax       AGR-agnostic min-max (Shejwalkar & Houmansadr 2021)\\n\\nRule-aware attacker (the study of this paper). It reads the PUBLIC part of the\\nacceptance rule from the chain and picks, by bisection, the largest push toward\\nits goal that the rule still accepts. It may also farm reputation first.\\nAttackers see only what a tenant could see: their own data, their own honest\\nupdates, the public rule state and (if the contract has no commit-reveal) the\\ncurrent round's benign updates.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport numpy as np\\nimport torch\\nfrom scipy.stats import norm\\n\\nfrom . import defenses as D\\n\\n\\nclass Attacker:\\n    def __init__(self, kind: str, mal_idx, n_clients: int, farm_rounds: int = 0, gamma_max: float = 4.0,\\n                 see_benign: bool = False, margin: float = 0.9, surrogate_n: int = 600, seed: int = 0):\\n        self.kind = kind\\n        self.mal = list(mal_idx)\\n        self.n = n_clients\\n        self.farm = farm_rounds\\n        self.gamma_max = gamma_max\\n        self.see_benign = see_benign\\n        self.margin = margin\\n        self.surrogate_n = surrogate_n\\n        self.rng = np.random.default_rng(seed)\\n        self.last_gamma = 0.0\\n        self.train = None          # dict(epochs, bs, lr, gen, device) set by run_one\\n        self.target = \\\"dos\\\"\\n        self.leak_cap = 8000       # how many leaked pool samples a memorizing attacker can use\\n        self._surr = None          # persistent stand-in copy of the defense (opt_adaptive)\\n        self._prev_a = None\\n        self.make_surrogate = None # factory(pool) -> a fresh defense of the same kind, set by run_one\\n        self.families = None\\n\\n    # --------------------------------------------------------------- helpers\\n    def _proxies(self, honest_U, mal_idx_t, benign_U=None):\\n        if self.see_benign and benign_U is not None:\\n            return benign_U\\n        return honest_U[mal_idx_t]\\n\\n    def _surrogate(self, tenants, defense, ctx):\\n        \\\"\\\"\\\"Attacker's stand-in for the validation probe: stratified samples of its own clean data\\n        (plus any leaked pool samples). Used only against sealed rules.\\\"\\\"\\\"\\n        Xs, fs = [], []\\n        for i in self.mal:\\n            t = tenants[i]\\n            Xs.append(t[\\\"X\\\"]); fs.append(t[\\\"fam\\\"])\\n        X = np.concatenate(Xs); fam = np.concatenate(fs)\\n        sel = []\\n        for f in np.unique(fam):\\n            ii = np.flatnonzero(fam == f)\\n            sel.append(self.rng.choice(ii, min(len(ii), max(1, self.surrogate_n // len(np.unique(fam)))), replace=False))\\n        sel = np.concatenate(sel)\\n        Xp, fp = X[sel], fam[sel]\\n        v = defense.public_view()\\n        if \\\"leak_idx\\\" in v:\\n            li = v[\\\"leak_idx\\\"][: max(self.surrogate_n, self.leak_cap if self.kind == \\\"probe_memorize\\\" else 0)]\\n            Xp = np.concatenate([Xp, v[\\\"pool_X\\\"][li].cpu().numpy()])\\n            fp = np.concatenate([fp, v[\\\"pool_fam\\\"][li]])\\n        return torch.as_tensor(Xp, device=ctx.theta.device), fp\\n\\n    # --------------------------------------------------------------- main\\n    @torch.no_grad()\\n    def craft(self, honest_U: torch.Tensor, poison_U: torch.Tensor, ctx, defense, tenants, benign_U=None):\\n        \\\"\\\"\\\"honest_U [C,P] honest updates of all clients; poison_U [m,P] updates trained on flipped\\n        target labels (rows follow self.mal). Returns submitted updates for the malicious clients [m,P].\\\"\\\"\\\"\\n        mi = torch.as_tensor(self.mal, device=honest_U.device)\\n        hu = honest_U[mi]\\n        k = self.kind\\n        if k == \\\"none\\\":\\n            return hu\\n        if k == \\\"targeted_lf\\\":\\n            return poison_U\\n        if k == \\\"signflip\\\":\\n            return -hu\\n        if k == \\\"lie\\\":\\n            mu, sd = hu.mean(0), hu.std(0) + 1e-12\\n            n, m = self.n, len(self.mal)\\n            s = int(np.floor(n / 2 + 1) - m)\\n            z = float(norm.ppf(min(max((n - s) / n, 0.5001), 0.9999)))\\n            return (mu - z * sd).unsqueeze(0).repeat(m, 1)\\n        if k == \\\"minmax\\\":\\n            prox = self._proxies(honest_U, mi, benign_U)\\n            mu = prox.mean(0)\\n            pdir = -mu / (mu.norm() + 1e-12)\\n            dmax = torch.cdist(prox, prox).max()\\n            lo, hi = 0.0, 50.0 * (mu.norm().item() + 1e-9)\\n            for _ in range(20):\\n                g = (lo + hi) / 2\\n                u = mu + g * pdir\\n                if (prox - u).norm(dim=1).max() <= dmax:\\n                    lo = g\\n                else:\\n                    hi = g\\n            self.last_gamma = lo\\n            return (mu + lo * pdir).unsqueeze(0).repeat(len(self.mal), 1)\\n        if k in (\\\"opt_adaptive\\\", \\\"opt_adaptive_x2\\\", \\\"opt_adaptive_x4\\\", \\\"opt_adaptive_x8\\\"):\\n            return self._opt_adaptive(honest_U, hu, poison_U, ctx, defense, tenants, mi, budget=int(k.split(\\\"_x\\\")[1]) if \\\"_x\\\" in k else 1)\\n        if k == \\\"opt_noisy\\\":       # imperfect knowledge of honest updates: stand-ins 4x noisier\\n            return self._opt_adaptive(honest_U, hu, poison_U, ctx, defense, tenants, mi, noise=2.0)\\n        if k == \\\"opt_lowdata\\\":     # one tenth of the attacker's own flows for its objective and its stand-in probe\\n            return self._opt_adaptive(honest_U, hu, poison_U, ctx, defense, tenants, mi, eval_n=150, surr_n=400)\\n        if k == \\\"opt_es\\\":          # independent search: evolution strategy over a six-direction space, same budget\\n            return self._opt_es(honest_U, hu, poison_U, ctx, defense, tenants, mi)\\n        if k == \\\"agr_tailored\\\":    # Shejwalkar & Houmansadr (NDSS 2021) AGR-tailored attack against the (simulated) rule\\n            return self._agr_tailored(honest_U, hu, poison_U, ctx, defense, tenants, mi)\\n        if k == \\\"probe_memorize\\\":\\n            if ctx.round < 0 or defense.name not in D.PROBE_NAMES:\\n                return self._rule_aware(honest_U, hu, poison_U, ctx, defense, tenants, benign_U, mi)\\n            u_mem = self._memorize(hu, poison_U, ctx, defense, tenants)\\n            return self._rule_aware(honest_U, hu, u_mem, ctx, defense, tenants, benign_U, mi)\\n        if k in (\\\"rule_aware\\\", \\\"rule_aware_farm\\\"):\\n            if k == \\\"rule_aware_farm\\\" and ctx.round < self.farm:\\n                self.last_gamma = 0.0\\n                return hu                                    # build reputation first\\n            return self._rule_aware(honest_U, hu, poison_U, ctx, defense, tenants, benign_U, mi)\\n        raise KeyError(k)\\n\\n    # --------------------------------------------------------------- defense-agnostic optimiser\\n    def _opt_adaptive(self, honest_U, hu, pu, ctx, defense, tenants, mi, budget: int = 1, noise: float = 0.5,\\n                      eval_n: int = 1500, surr_n: int = 4000):\\n        \\\"\\\"\\\"End-to-end adaptive attack that works against any rule.\\n\\n        The attacker keeps a private stand-in copy of the defense (same rule, same constants, same public state, but\\n        its OWN probe pool: the exact probe if the rule is public, a stand-in built from its own data otherwise).\\n        Each round it builds a stand-in population (its own honest updates, resampled with noise, as the benign\\n        tenants), then searches over a small family of update shapes\\n            u_i = h_i + a1 (p_i - h_i) + a2 (-h_i) + a3 (-unit(mean h) |h_i|)\\n        and keeps the one that does the most damage to the AGGREGATE produced by the stand-in defense, measured as\\n        the drop in DoS recall on its own DoS samples (with a small penalty for extra false alarms).\\n        No assumption about the rule is built in, so the same code attacks median, Krum, CosRep, norm gates and\\n        probe rules alike.\\\"\\\"\\\"\\n        import copy\\n        from . import model as M\\n        dev, m, N = hu.device, len(self.mal), self.n\\n        rng = self.rng\\n        X_own = np.concatenate([tenants[i][\\\"X\\\"] for i in self.mal]); F_own = np.concatenate([tenants[i][\\\"fam\\\"] for i in self.mal])\\n        ti = np.flatnonzero(F_own == self.target); bi = np.flatnonzero(F_own == \\\"benign\\\")\\n        if len(ti) == 0 or len(bi) == 0:\\n            return hu\\n        ti = rng.choice(ti, min(len(ti), eval_n), replace=False); bi = rng.choice(bi, min(len(bi), eval_n), replace=False)\\n        Xt, Xb = torch.as_tensor(X_own[ti], device=dev), torch.as_tensor(X_own[bi], device=dev)\\n\\n        if self._surr is None:                                   # build the private stand-in defense once\\n            pool = None\\n            if defense.name in D.PROBE_NAMES:\\n                v = defense.public_view()\\n                if defense.public:\\n                    idx = v[\\\"probe_idx\\\"]\\n                    Kx, Kf = v[\\\"pool_X\\\"][idx].cpu().numpy(), np.asarray(v[\\\"pool_fam\\\"])[idx]\\n                else:\\n                    old = self.surrogate_n; self.surrogate_n = surr_n\\n                    Xp, Kf = self._surrogate(tenants, defense, ctx); self.surrogate_n = old\\n                    Kx, Kf = Xp.cpu().numpy(), np.asarray(Kf)\\n                pool = {\\\"X\\\": Kx, \\\"y\\\": (Kf != \\\"benign\\\").astype(np.int64), \\\"fam\\\": Kf}\\n            self._surr = self.make_surrogate(pool)\\n        surr = self._surr\\n        for a in (\\\"rep\\\", \\\"trust\\\", \\\"public_thr\\\", \\\"ref_norm\\\", \\\"tau\\\", \\\"banned\\\"):   # public on-chain state is readable\\n            if hasattr(defense, a) and hasattr(surr, a):\\n                val = getattr(defense, a)\\n                setattr(surr, a, val.clone() if torch.is_tensor(val) else val)\\n\\n        honest_idx = [i for i in range(N) if i not in set(self.mal)]\\n        mu = hu.mean(0); sd = hu.std(0) if m > 1 else hu.abs() * 0.2\\n        sd = torch.clamp(sd, min=1e-8)\\n        g = torch.Generator(device=dev).manual_seed(int(rng.integers(1 << 30)))\\n        prox = hu[torch.randint(0, m, (len(honest_idx),), generator=g, device=dev)] + noise * sd * torch.randn(len(honest_idx), hu.shape[1], generator=g, device=dev)\\n        mdir = -mu / (mu.norm() + 1e-12)\\n        from .defenses import Ctx\\n        cs = Ctx(ctx.round, ctx.theta, ctx.shapes, N, ctx.beacon)\\n\\n        def build(a):\\n            U = torch.zeros(N, hu.shape[1], device=dev)\\n            U[honest_idx] = prox\\n            U[self.mal] = hu + a[0] * (pu - hu) + a[1] * (-hu) + a[2] * mdir * hu.norm(dim=1, keepdim=True)\\n            return U\\n\\n        def score(a):\\n            sc = copy.deepcopy(surr)\\n            delta = sc.aggregate(build(a), cs)\\n            th = ctx.theta + delta\\n            pt = M.predict(th, ctx.shapes, Xt)[0]; pb = M.predict(th, ctx.shapes, Xb)[0]\\n            return float((pt == 1).float().mean()), float((pb == 1).float().mean())\\n\\n        rec0, fpr0 = score((0.0, 0.0, 0.0))\\n        cands = [(1, 0, 0), (2, 0, 0), (4, 0, 0), (8, 0, 0), (1, 1, 0), (2, 1, 0), (0, 1, 0), (0, 0, 1), (0, 0, 2), (1, 0, 1), (2, 0, 1)]\\n        if self._prev_a is not None:\\n            cands.append(tuple(self._prev_a))\\n        for _ in range(12 * budget):\\n            a = np.exp(rng.uniform(np.log(0.05), np.log(8.0), size=3)) * (rng.random(3) < 0.6)\\n            cands.append(tuple(float(x) for x in a))\\n        best, bj = (0.0, 0.0, 0.0), 0.0\\n\\n        def consider(a):\\n            nonlocal best, bj\\n            rec, fpr = score(a)\\n            j = (rec0 - rec) - 0.5 * max(0.0, fpr - fpr0 - 0.02)\\n            if j > bj:\\n                best, bj = a, j\\n        for a in cands:\\n            consider(a)\\n        if budget > 1:                                           # local refinement around the incumbent\\n            for _ in range(12 * (budget - 1)):\\n                a = tuple(float(max(0.0, x * np.exp(rng.normal(0, 0.35)) + (0.05 * rng.random() if x == 0 and rng.random() < 0.2 else 0.0))) for x in best)\\n                consider(a)\\n        self._prev_a = best\\n        self.last_gamma = float(best[0])\\n        U_best = build(best)\\n        surr.aggregate(U_best, cs)                              # advance the stand-in's own state\\n        return U_best[self.mal]\\n\\n    # --------------------------------------------------------------- independent attackers (R3)\\n    def _sim_setup(self, hu, ctx, defense, tenants, eval_n=1500, surr_n=4000, noise=0.5):\\n        \\\"\\\"\\\"Shared set-up for the R3 attackers: own evaluation flows, private copy of the published rule (with public state\\n        copied from chain), stand-ins for the honest tenants. Mirrors _opt_adaptive's set-up; kept separate so that the\\n        pre-registered optimiser path stays byte-for-byte unchanged.\\\"\\\"\\\"\\n        import copy\\n        from . import model as M\\n        from .defenses import Ctx\\n        dev, m, N = hu.device, len(self.mal), self.n\\n        rng = self.rng\\n        X_own = np.concatenate([tenants[i][\\\"X\\\"] for i in self.mal]); F_own = np.concatenate([tenants[i][\\\"fam\\\"] for i in self.mal])\\n        ti = np.flatnonzero(F_own == self.target); bi = np.flatnonzero(F_own == \\\"benign\\\")\\n        if len(ti) == 0 or len(bi) == 0:\\n            return None\\n        ti = rng.choice(ti, min(len(ti), eval_n), replace=False); bi = rng.choice(bi, min(len(bi), eval_n), replace=False)\\n        Xt, Xb = torch.as_tensor(X_own[ti], device=dev), torch.as_tensor(X_own[bi], device=dev)\\n        if self._surr is None:\\n            pool = None\\n            if defense.name in D.PROBE_NAMES:\\n                v = defense.public_view()\\n                if defense.public:\\n                    idx = v[\\\"probe_idx\\\"]; Kx, Kf = v[\\\"pool_X\\\"][idx].cpu().numpy(), np.asarray(v[\\\"pool_fam\\\"])[idx]\\n                else:\\n                    old = self.surrogate_n; self.surrogate_n = surr_n\\n                    Xp, Kf = self._surrogate(tenants, defense, ctx); self.surrogate_n = old\\n                    Kx, Kf = Xp.cpu().numpy(), np.asarray(Kf)\\n                pool = {\\\"X\\\": Kx, \\\"y\\\": (Kf != \\\"benign\\\").astype(np.int64), \\\"fam\\\": Kf}\\n            self._surr = self.make_surrogate(pool)\\n        surr = self._surr\\n        for a in (\\\"rep\\\", \\\"trust\\\", \\\"public_thr\\\", \\\"ref_norm\\\", \\\"tau\\\", \\\"banned\\\"):\\n            if hasattr(defense, a) and hasattr(surr, a):\\n                val = getattr(defense, a); setattr(surr, a, val.clone() if torch.is_tensor(val) else val)\\n        honest_idx = [i for i in range(N) if i not in set(self.mal)]\\n        mu = hu.mean(0); sd = torch.clamp(hu.std(0) if m > 1 else hu.abs() * 0.2, min=1e-8)\\n        g = torch.Generator(device=dev).manual_seed(int(rng.integers(1 << 30)))\\n        prox = hu[torch.randint(0, m, (len(honest_idx),), generator=g, device=dev)] + noise * sd * torch.randn(len(honest_idx), hu.shape[1], generator=g, device=dev)\\n        cs = Ctx(ctx.round, ctx.theta, ctx.shapes, N, ctx.beacon)\\n\\n        def build(mal_rows):\\n            U = torch.zeros(N, hu.shape[1], device=dev); U[honest_idx] = prox; U[self.mal] = mal_rows\\n            return U\\n\\n        def aggregate(mal_rows):\\n            return copy.deepcopy(surr).aggregate(build(mal_rows), cs)\\n\\n        def recall_fpr(delta):\\n            th = ctx.theta + delta\\n            return float((M.predict(th, ctx.shapes, Xt)[0] == 1).float().mean()), float((M.predict(th, ctx.shapes, Xb)[0] == 1).float().mean())\\n        return dict(surr=surr, cs=cs, build=build, aggregate=aggregate, recall_fpr=recall_fpr, mu=mu, sd=sd, N=N, m=m)\\n\\n    def _opt_es(self, honest_U, hu, pu, ctx, defense, tenants, mi, gens: int = 4, pop: int = 6):\\n        \\\"\\\"\\\"Independent search: a (mu, lambda) evolution strategy over SIX update directions (poison push, sign flip, mean shift,\\n        inverse standard deviation, inverse sign, LIE target), same objective as _opt_adaptive and the same number of\\n        evaluations of the private rule copy (gens x pop = 24).\\\"\\\"\\\"\\n        S = self._sim_setup(hu, ctx, defense, tenants)\\n        if S is None:\\n            return hu\\n        rng = self.rng; mu, sd = S[\\\"mu\\\"], S[\\\"sd\\\"]; nh = hu.norm(dim=1, keepdim=True)\\n        n_, m_ = self.n, len(self.mal); s_ = int(np.floor(n_ / 2 + 1) - m_)\\n        z = float(norm.ppf(min(max((n_ - s_) / n_, 0.5001), 0.9999)))\\n        dirs = [pu - hu, -hu, (-mu / (mu.norm() + 1e-12)) * nh, (-sd / (sd.norm() + 1e-12)) * nh,\\n                (-torch.sign(mu) / np.sqrt(mu.numel())) * nh, (mu - z * sd).unsqueeze(0) - hu]\\n        rows = lambda a: hu + sum(float(a[k]) * dirs[k] for k in range(6))\\n        rec0, fpr0 = S[\\\"recall_fpr\\\"](S[\\\"aggregate\\\"](hu))\\n\\n        def J(a):\\n            rec, fpr = S[\\\"recall_fpr\\\"](S[\\\"aggregate\\\"](rows(a)))\\n            return (rec0 - rec) - 0.5 * max(0.0, fpr - fpr0 - 0.02)\\n        mean = np.array(self._prev_a if (self._prev_a is not None and len(self._prev_a) == 6) else np.zeros(6), dtype=float)\\n        sigma, best, bj = 1.0, np.zeros(6), 0.0\\n        for _ in range(gens):\\n            cand = [np.clip(mean + sigma * rng.normal(size=6), 0.0, 8.0) for _ in range(pop)]\\n            scores = [J(a) for a in cand]\\n            order = np.argsort(scores)[::-1]\\n            if scores[order[0]] > bj:\\n                best, bj = cand[order[0]], scores[order[0]]\\n            mean = np.mean([cand[i] for i in order[:2]] + [best], axis=0); sigma *= 0.6\\n        self._prev_a = tuple(float(x) for x in best); self.last_gamma = float(best[0])\\n        out = rows(best); S[\\\"surr\\\"].aggregate(S[\\\"build\\\"](out), S[\\\"cs\\\"])\\n        return out\\n\\n    def _agr_tailored(self, honest_U, hu, pu, ctx, defense, tenants, mi):\\n        \\\"\\\"\\\"AGR-tailored attack of Shejwalkar and Houmansadr (NDSS 2021) with partial knowledge, run against the attacker's\\n        private copy of the published rule: every malicious tenant submits mu + gamma * p for a perturbation p in\\n        {inverse unit vector, inverse standard deviation, inverse sign}; gamma and p are chosen to push the aggregate as far\\n        as possible from the benign aggregate (untargeted objective, as in the original). 3 directions x 8 gammas = 24.\\\"\\\"\\\"\\n        S = self._sim_setup(hu, ctx, defense, tenants)\\n        if S is None:\\n            return hu\\n        mu, sd = S[\\\"mu\\\"], S[\\\"sd\\\"]; m_ = len(self.mal)\\n        ref = S[\\\"aggregate\\\"](hu)\\n        P = [(-mu / (mu.norm() + 1e-12), float(mu.norm())), (-sd, 1.0), (-torch.sign(mu) * float(mu.norm()) / np.sqrt(mu.numel()), 1.0)]\\n        best, bd = hu, -1.0\\n        for p, g0 in P:\\n            for k in range(-3, 5):\\n                u = (mu + g0 * (2.0 ** k) * p).unsqueeze(0).repeat(m_, 1)\\n                dv = float((S[\\\"aggregate\\\"](u) - ref).norm())\\n                if dv > bd:\\n                    best, bd = u, dv\\n        S[\\\"surr\\\"].aggregate(S[\\\"build\\\"](best), S[\\\"cs\\\"])\\n        return best\\n\\n    # --------------------------------------------------------------- teach-to-the-test\\n    def _memorize(self, hu, pu, ctx, defense, tenants):\\n        \\\"\\\"\\\"Poison, then patch the probe.\\n\\n        The attacker trains on its own data with target-family flows relabelled benign (the poison) and ALSO on the\\n        probe samples it knows, with their TRUE labels, for longer than an honest tenant. Against a public rule the\\n        known probe is the exact one, so the update can keep the probe check clean while still suppressing DoS flows\\n        that are not in the probe. Against a sealed rule it knows only a stand-in (its own samples plus any leaked\\n        pool samples), so the patch does not transfer to the probe drawn at reveal time.\\\"\\\"\\\"\\n        from . import model as M\\n\\n        v = defense.public_view()\\n        if defense.public:\\n            idx = v[\\\"probe_idx\\\"]\\n            Kx, Kf = v[\\\"pool_X\\\"][idx], v[\\\"pool_fam\\\"][idx]\\n        else:\\n            Kx, Kf = self._surrogate(tenants, defense, ctx)\\n        dev = ctx.theta.device\\n        Kx = Kx.to(dev) if torch.is_tensor(Kx) else torch.as_tensor(Kx, device=dev)\\n        Ky = torch.as_tensor((np.asarray(Kf) != \\\"benign\\\").astype(np.int64), device=dev)\\n        Xs, Ys = [], []\\n        rep = max(1, int(round(0.5 * len(tenants[self.mal[0]][\\\"y\\\"]) / max(len(Ky), 1))))\\n        for i in self.mal:\\n            t = tenants[i]\\n            x = torch.as_tensor(t[\\\"X\\\"], device=dev)\\n            y = torch.as_tensor(t[\\\"y\\\"], device=dev).clone()\\n            y[torch.as_tensor(t[\\\"fam\\\"] == self.target, device=dev)] = 0          # the poison\\n            Xs.append(torch.cat([x] + [Kx] * rep)); Ys.append(torch.cat([y] + [Ky] * rep))\\n        X, Y = torch.stack(Xs), torch.stack(Ys)\\n        tr = self.train\\n        start = ctx.theta + pu.mean(0)                                           # begin from the poisoned model\\n        with torch.enable_grad():                                                # craft() runs under no_grad\\n            th = M.local_train(start, ctx.shapes, X, Y, max(8, 4 * tr[\\\"epochs\\\"]), tr[\\\"bs\\\"], tr[\\\"lr\\\"], tr[\\\"gen\\\"])\\n        return th - ctx.theta.unsqueeze(0)\\n\\n    # --------------------------------------------------------------- rule-aware search\\n    def _rule_aware(self, honest_U, hu, pu, ctx, defense, tenants, benign_U, mi):\\n        d = pu - hu                                           # push toward the goal\\n        ref = self._proxies(honest_U, mi, benign_U).mean(0)   # attacker's estimate of the benign mean\\n        name = defense.name\\n\\n        if name in D.PROBE_NAMES:\\n            v = defense.public_view()\\n            if defense.public:                                # exact probe is readable on chain\\n                idx = v[\\\"probe_idx\\\"]\\n                Xp, fp = v[\\\"pool_X\\\"][idx], v[\\\"pool_fam\\\"][idx]\\n            else:                                             # sealed: only a surrogate from its own data (+ any leaked samples)\\n                Xp, fp = self._surrogate(tenants, defense, ctx)\\n            mode = defense.mode\\n            if mode == \\\"abs\\\":\\n                thr = defense.public_thr                      # published absolute tolerance\\n\\n                def passes(gam):\\n                    dmg, _ = D.damage(ctx.theta, hu + gam * d, ctx.shapes, Xp, fp, defense.families)\\n                    return bool((dmg <= self.margin * thr).all())\\n            elif mode == \\\"rel_max\\\":\\n                # strongest realistic attacker: it knows the rule FORM and its constants (kappa, floor) because they\\n                # are on chain, and estimates the benign median/MAD of harm from its own honest updates.\\n                d0, _ = D.damage(ctx.theta, hu, ctx.shapes, Xp, fp, defense.families)\\n                med = d0.median(); mad = (d0 - med).abs().median()\\n                thr = float(max(med + defense.kappa * (mad + 1e-3) + defense.delta_floor, d0.max()))\\n\\n                def passes(gam):\\n                    dmg, _ = D.damage(ctx.theta, hu + gam * d, ctx.shapes, Xp, fp, defense.families)\\n                    return bool((dmg <= thr).all())\\n            else:                                             # rel_fam: peer test per check\\n                Rh, fh = D.probe_stats(ctx.theta, hu, ctx.shapes, Xp, fp, defense.families)\\n                _, ref = D.peer_z(Rh[1:], fh[1:], defense.s_min)   # its own honest updates stand in for the peers\\n\\n                def passes(gam):\\n                    R, f_ = D.probe_stats(ctx.theta, hu + gam * d, ctx.shapes, Xp, fp, defense.families)\\n                    z, _ = D.peer_z(R[1:], f_[1:], defense.s_min, ref=ref)\\n                    return bool((z.max(1).values <= self.margin * defense.kappa).all())\\n        elif name == \\\"normgate\\\":\\n            tau = defense.public_view()[\\\"tau\\\"]\\n            if not np.isfinite(tau):\\n                tau = 1.5 * ref.norm().item()\\n\\n            def passes(gam):\\n                return bool(((hu + gam * d).norm(dim=1) <= self.margin * tau).all())\\n        elif name == \\\"cosrep\\\":\\n            def passes(gam):\\n                u = hu + gam * d\\n                return bool((D._cos(u, ref) > 0.35).all())    # stays well above the reputation floor\\n        elif name == \\\"trustaware\\\":\\n            prox = self._proxies(honest_U, mi, benign_U)\\n            dmax = (prox - ref).norm(dim=1).max()\\n\\n            def passes(gam):\\n                return bool(((hu + gam * d) - ref).norm(dim=1).max() <= self.margin * dmax * 1.5)\\n        else:                                                  # fedavg / median / trmean / mkrum\\n            prox = self._proxies(honest_U, mi, benign_U)\\n            dmax = torch.cdist(prox, prox).max()\\n\\n            def passes(gam):\\n                u = hu + gam * d\\n                return bool((u.unsqueeze(1) - prox.unsqueeze(0)).norm(dim=2).max() <= dmax * 1.0)\\n\\n        lo, hi = 0.0, self.gamma_max\\n        if passes(hi):\\n            lo = hi\\n        else:\\n            for _ in range(14):\\n                g = (lo + hi) / 2\\n                if passes(g):\\n                    lo = g\\n                else:\\n                    hi = g\\n        self.last_gamma = lo\\n        return hu + lo * d\\n\", \"data.py\": \"\\\"\\\"\\\"Data loading, cleaning and tenant partitioning.\\n\\nSources are the NetFlow-v2 versions of four public IDS datasets. They share one\\nfeature schema, so each source can stand in for a different cloud tenant type.\\nEverything here is schema-tolerant: label columns are detected by name and\\nnon-numeric / identifier columns are dropped, because the Kaggle copies have\\nnot been opened on the author's machine yet. The inspection notebook prints\\nthe real schema before any experiment runs.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport glob\\nimport hashlib\\nimport json\\nimport os\\nimport re\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport pandas as pd\\n\\n# Kaggle slugs checked against the Kaggle public API on 2026-10-14.\\nKAGGLE_SOURCES = {\\n    \\\"unsw\\\": \\\"dhoogla/nfunswnb15v2\\\",\\n    \\\"cse18\\\": \\\"dhoogla/nfcsecicids2018v2\\\",\\n    \\\"ton\\\": \\\"dhoogla/nftoniotv2\\\",\\n    \\\"bot\\\": \\\"dhoogla/nfbotiotv2\\\",\\n}\\nEXTRA_SOURCES = {\\n    \\\"ciciot\\\": \\\"himadri07/ciciot2023\\\",\\n    \\\"kube\\\": \\\"redamorsli/kube-ids0\\\",\\n}\\n\\nLABEL_COLS = (\\\"label\\\", \\\"attack\\\", \\\"attack_cat\\\", \\\"class\\\", \\\"target\\\")\\nDROP_PATTERNS = re.compile(r\\\"(ADDR|_PORT$|^SRC_|^DST_IP|DNS_QUERY_ID|^ID$|^FLOW_ID|TIMESTAMP|^TIME)\\\", re.I)\\n\\nFAMILIES = [\\\"benign\\\", \\\"dos\\\", \\\"recon\\\", \\\"bruteforce\\\", \\\"web\\\", \\\"exploit\\\", \\\"persist\\\", \\\"other\\\"]\\n_FAMILY_RULES = [\\n    (\\\"benign\\\", r\\\"^(benign|normal|0)$\\\"),\\n    (\\\"dos\\\", r\\\"ddos|dos|slowloris|hulk|goldeneye|slowhttp|loic|hoic\\\"),\\n    (\\\"recon\\\", r\\\"scan|reconnaissance|recon|probe\\\"),\\n    (\\\"bruteforce\\\", r\\\"brute|password|ssh|ftp\\\"),\\n    (\\\"web\\\", r\\\"injection|xss|sql|web\\\"),\\n    (\\\"exploit\\\", r\\\"exploit|shellcode|generic|analysis|worm|fuzz|ransom\\\"),\\n    (\\\"persist\\\", r\\\"backdoor|bot|infil|theft|mitm\\\"),\\n]\\n\\n\\ndef family_of(name) -> str:\\n    s = str(name).strip().lower()\\n    for fam, pat in _FAMILY_RULES:\\n        if re.search(pat, s):\\n            return fam\\n    return \\\"other\\\"\\n\\n\\ndef download(key: str) -> str:\\n    \\\"\\\"\\\"Download a Kaggle dataset with kagglehub and return its local path.\\\"\\\"\\\"\\n    import kagglehub\\n\\n    slug = {**KAGGLE_SOURCES, **EXTRA_SOURCES}[key]\\n    return kagglehub.dataset_download(slug)\\n\\n\\ndef _files(path: str):\\n    out = []\\n    for ext in (\\\"parquet\\\", \\\"csv\\\"):\\n        out += glob.glob(os.path.join(path, \\\"**\\\", f\\\"*.{ext}\\\"), recursive=True)\\n    return sorted(out)\\n\\n\\ndef _has_label(path: str) -> bool:\\n    \\\"\\\"\\\"True if the file is a data file (has a Label / Attack column), not a documentation file.\\\"\\\"\\\"\\n    try:\\n        if path.endswith(\\\".parquet\\\"):\\n            import pyarrow.parquet as pq\\n            cols = pq.ParquetFile(path).schema_arrow.names\\n        else:\\n            cols = pd.read_csv(path, nrows=0).columns.tolist()\\n    except Exception:\\n        return False\\n    return any(c.lower() in LABEL_COLS for c in cols)\\n\\n\\ndef _detect_labels(cols):\\n    low = {c.lower(): c for c in cols}\\n    lab = next((low[c] for c in (\\\"label\\\",) if c in low), None)\\n    att = next((low[c] for c in (\\\"attack\\\", \\\"attack_cat\\\", \\\"class\\\") if c in low), None)\\n    return lab, att\\n\\n\\ndef _rss_gb() -> float:\\n    try:\\n        import psutil\\n        return psutil.Process().memory_info().rss / 1e9\\n    except Exception:\\n        return float(\\\"nan\\\")\\n\\n\\ndef _keep_col(c: str) -> bool:\\n    \\\"\\\"\\\"Columns worth reading at all: labels, plus anything that is not an address/port/id column.\\\"\\\"\\\"\\n    return c.lower() in LABEL_COLS or not DROP_PATTERNS.search(c)\\n\\n\\ndef _iter_batches(files, batch_rows=100_000):\\n    \\\"\\\"\\\"Yield small pandas frames (only the needed columns) so memory stays flat.\\\"\\\"\\\"\\n    for f in files:\\n        if f.endswith(\\\".parquet\\\"):\\n            import pyarrow.parquet as pq\\n\\n            pf = pq.ParquetFile(f)\\n            cols = [c for c in pf.schema_arrow.names if _keep_col(c)]\\n            for b in pf.iter_batches(batch_size=batch_rows, columns=cols, use_threads=False):\\n                yield b.to_pandas()\\n        else:\\n            for ch in pd.read_csv(f, chunksize=batch_rows, low_memory=False,\\n                                  usecols=lambda c: _keep_col(c)):\\n                yield ch\\n\\n\\ndef load_source(path: str, max_rows: int = 400_000, seed: int = 0, per_family_cap: int = 80_000,\\n                batch_rows: int = 100_000, verbose: bool = True, dedupe: bool = True):\\n    \\\"\\\"\\\"Stream a dataset with flat memory; keep a family-balanced sample; clean and dedupe.\\n\\n    Each batch is turned into float32 arrays and only a thin random slice of every family is kept, so\\n    memory is set by the sample (about per_family_cap x families rows), not by the file size.\\n    Returns X (float32), y (0/1), fam (str array), feature names, and a report dict.\\n    \\\"\\\"\\\"\\n    import gc\\n\\n    rng = np.random.default_rng(seed)\\n    all_files = _files(path)\\n    files = [f for f in all_files if _has_label(f)]\\n    skipped = [os.path.basename(f) for f in all_files if f not in files]\\n    if not files:\\n        raise FileNotFoundError(f\\\"no data file with a Label/Attack column under {path} (saw {skipped})\\\")\\n    n_total = 0\\n    for f in files:\\n        if f.endswith(\\\".parquet\\\"):\\n            import pyarrow.parquet as pq\\n            n_total += pq.ParquetFile(f).metadata.num_rows\\n    n_batches = max(1, int(np.ceil(n_total / batch_rows))) if n_total else 50\\n    # keep ~2x the final cap per family across all batches, then trim at the end\\n    per_batch_cap = max(20, int(np.ceil(2.0 * per_family_cap / n_batches)))\\n    rep = {\\\"files\\\": [os.path.basename(f) for f in files], \\\"skipped_non_data_files\\\": skipped,\\n           \\\"rows_seen\\\": 0, \\\"n_batches\\\": n_batches}\\n    feat, Xs, fams = None, [], []\\n    for i, df in enumerate(_iter_batches(files, batch_rows)):\\n        rep[\\\"rows_seen\\\"] += len(df)\\n        lab, att = _detect_labels(df.columns)\\n        if lab is None and att is None:\\n            raise ValueError(f\\\"no label column in {df.columns.tolist()[:10]}\\\")\\n        if att is not None:\\n            names = df[att].astype(str)\\n            uniq = names.unique()\\n            m = {u: family_of(u) for u in uniq}\\n            fam = names.map(m).to_numpy(dtype=object)\\n        else:\\n            fam = np.where(df[lab].to_numpy() == 0, \\\"benign\\\", \\\"other\\\").astype(object)\\n        if feat is None:\\n            feat = [c for c in df.columns if c.lower() not in LABEL_COLS and not DROP_PATTERNS.search(c)\\n                    and pd.api.types.is_numeric_dtype(df[c])]\\n        X = df[feat].to_numpy(dtype=np.float32, copy=False)\\n        del df\\n        keep = []\\n        for f_ in np.unique(fam):\\n            ii = np.flatnonzero(fam == f_)\\n            if len(ii) > per_batch_cap:\\n                ii = rng.choice(ii, per_batch_cap, replace=False)\\n            keep.append(ii)\\n        keep = np.concatenate(keep)\\n        Xs.append(X[keep].copy())\\n        fams.append(fam[keep])\\n        del X, fam\\n        if verbose and (i % 25 == 0):\\n            print(f\\\"   batch {i + 1}/{n_batches}  kept so far {sum(len(a) for a in Xs):,} rows  rss {_rss_gb():.1f} GB\\\", flush=True)\\n            gc.collect()\\n    X = np.concatenate(Xs); fam = np.concatenate(fams)\\n    del Xs, fams\\n    gc.collect()\\n    ok = np.isfinite(X).all(axis=1)\\n    X, fam = X[ok], fam[ok]\\n    n0 = len(X)\\n    key = pd.util.hash_pandas_object(\\n        pd.concat([pd.DataFrame(X), pd.Series(fam, name=\\\"fam\\\")], axis=1), index=False).to_numpy()\\n    _, first = np.unique(key, return_index=True)\\n    first = np.sort(first) if dedupe else np.arange(len(key))   # dedupe=False keeps exact duplicates (inflation experiment)\\n    X, fam = X[first], fam[first]\\n    rep.update(rows_after_nan=int(n0), rows_after_dedupe=int(len(X)),\\n               dup_frac=float(1 - len(X) / max(n0, 1)), n_features=len(feat))\\n    # constant columns carry no signal\\n    nz = X.std(axis=0) > 0\\n    X = X[:, nz]\\n    rep[\\\"constant_features_removed\\\"] = [c for c, k in zip(feat, nz) if not k]\\n    feat = [c for c, k in zip(feat, nz) if k]\\n    rep[\\\"n_features_final\\\"] = len(feat)\\n    # family-balanced cap\\n    keep = np.zeros(len(X), dtype=bool)\\n    for f_ in np.unique(fam):\\n        ii = np.flatnonzero(fam == f_)\\n        if len(ii) > per_family_cap:\\n            ii = rng.choice(ii, per_family_cap, replace=False)\\n        keep[ii] = True\\n    X, fam = X[keep], fam[keep]\\n    if len(X) > max_rows:\\n        idx = rng.choice(len(X), max_rows, replace=False)\\n        X, fam = X[idx], fam[idx]\\n    y = (fam != \\\"benign\\\").astype(np.int64)\\n    rep[\\\"family_counts\\\"] = {str(k): int(v) for k, v in zip(*np.unique(fam, return_counts=True))}\\n    rep[\\\"rows_final\\\"] = int(len(X))\\n    return X.astype(np.float32), y, fam.astype(str), feat, rep\\n\\n\\ndef squash(X: np.ndarray) -> np.ndarray:\\n    \\\"\\\"\\\"Sign-preserving log transform for heavy-tailed flow features.\\\"\\\"\\\"\\n    return np.sign(X) * np.log1p(np.abs(X))\\n\\n\\n@dataclass\\nclass Scaler:\\n    mean: np.ndarray\\n    std: np.ndarray\\n\\n    @classmethod\\n    def fit(cls, X):\\n        Z = squash(X)\\n        return cls(Z.mean(0), Z.std(0) + 1e-6)\\n\\n    def __call__(self, X):\\n        return np.clip((squash(X) - self.mean) / self.std, -5, 5).astype(np.float32)\\n\\n\\n@dataclass\\nclass World:\\n    \\\"\\\"\\\"Everything one experiment needs.\\\"\\\"\\\"\\n    n_features: int\\n    tenants: list          # list of dict(X, y, fam, source)\\n    test: dict             # source -> dict(X, y, fam)\\n    audit: dict            # source -> dict(X, y, fam)  pool for probes (disjoint from train/test)\\n    sources: list\\n    meta: dict = field(default_factory=dict)\\n\\n\\ndef build_world(pools: dict, n_tenants: int = 20, n_train: int = 3000, n_test: int = 6000,\\n                n_audit: int = 8000, alpha: float = 1.0, seed: int = 0,\\n                holdout_source: str | None = None, scaler_fit: str = \\\"train\\\") -> World:\\n    \\\"\\\"\\\"pools: source -> (X, y, fam) of RAW features. Splits into tenants / test / audit.\\n\\n    Tenants are spread evenly over the training sources. Within a source the class\\n    mix of each tenant is drawn from Dirichlet(alpha * global mix) for label skew.\\n    \\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    sources = [s for s in pools if s != holdout_source]\\n    # Split first (raw indices), then fit the scaler on the tenants' TRAINING rows only and freeze it for the probe pool\\n    # and the test sets. Fitting on the union of tenant rows equals aggregating per-tenant feature means and variances,\\n    # which a consortium can do without sharing flows. (scaler_fit=\\\"all\\\" reproduces the earlier, leaky behaviour.)\\n    test_idx, audit_idx, left = {}, {}, {}\\n    for s, (X, y, fam) in pools.items():\\n        idx = rng.permutation(len(X))\\n        nt = min(n_test, len(X) // 4)\\n        na = min(n_audit, len(X) // 4)\\n        test_idx[s], audit_idx[s], left[s] = idx[:nt], idx[nt:nt + na], idx[nt + na:]\\n    per = max(1, n_tenants // len(sources))\\n    tenant_sel = []\\n    for s in sources:\\n        X, y, fam = pools[s]\\n        rest = left[s]\\n        fams = np.unique(fam[rest])\\n        base = np.array([np.mean(fam[rest] == f) for f in fams])\\n        base = np.maximum(base, 0.02)\\n        base /= base.sum()\\n        by = {f: rng.permutation(rest[fam[rest] == f]) for f in fams}\\n        ptr = {f: 0 for f in fams}\\n        for _ in range(per):\\n            p = rng.dirichlet(alpha * len(fams) * base + 0.05)\\n            cnt = np.maximum(1, np.round(p * n_train).astype(int))\\n            sel = []\\n            for f, c in zip(fams, cnt):\\n                pool_f = by[f]\\n                take = pool_f[np.arange(ptr[f], ptr[f] + c) % len(pool_f)]\\n                ptr[f] += c\\n                sel.append(take)\\n            sel = np.concatenate(sel)\\n            sel = rng.permutation(sel)[:n_train]\\n            if len(sel) < n_train:  # pad by resampling\\n                sel = np.concatenate([sel, rng.choice(sel, n_train - len(sel))])\\n            tenant_sel.append((s, sel))\\n    if scaler_fit == \\\"all\\\":\\n        scaler = Scaler.fit(np.concatenate([pools[s][0] for s in sources]))\\n    else:\\n        train_rows = np.concatenate([pools[s][0][np.unique(sel)] for s, sel in tenant_sel])\\n        scaler = Scaler.fit(train_rows)\\n    tenants = [dict(X=scaler(pools[s][0][sel]), y=pools[s][1][sel], fam=pools[s][2][sel], source=s) for s, sel in tenant_sel]\\n    test, audit = {}, {}\\n    for s, (X, y, fam) in pools.items():\\n        test[s] = dict(X=scaler(X[test_idx[s]]), y=y[test_idx[s]], fam=fam[test_idx[s]])\\n        if s != holdout_source:          # a held-out source must not leak into the probe pool\\n            audit[s] = dict(X=scaler(X[audit_idx[s]]), y=y[audit_idx[s]], fam=fam[audit_idx[s]])\\n    return World(n_features=pools[sources[0]][0].shape[1], tenants=tenants, test=test, audit=audit, sources=sources,\\n                 meta=dict(alpha=alpha, n_tenants=len(tenants), n_train=n_train, holdout=holdout_source, scaler_fit=scaler_fit))\\n\\n\\ndef split_pools(pools: dict, seed: int = 12345):\\n    \\\"\\\"\\\"Split every source into two DISJOINT halves, stratified by family: one only for tuning\\n    hyper-parameters, one only for the final evaluation. Rows never appear in both.\\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    tune, final = {}, {}\\n    for s, (X, y, fam) in pools.items():\\n        a = np.zeros(len(X), dtype=bool)\\n        for f in np.unique(fam):\\n            ii = rng.permutation(np.flatnonzero(fam == f))\\n            a[ii[: len(ii) // 2]] = True\\n        tune[s] = (X[a], y[a], fam[a])\\n        final[s] = (X[~a], y[~a], fam[~a])\\n    return tune, final\\n\\n\\nKUBE_LABELS = {0: \\\"benign\\\", 1: \\\"dos\\\", 2: \\\"dos\\\", 3: \\\"bruteforce\\\", 4: \\\"web\\\"}   # per the dataset README (DVWA / Bank of Anthos)\\n\\n\\ndef load_kube(path: str, max_rows: int = 200_000, seed: int = 0, verbose: bool = True):\\n    \\\"\\\"\\\"Kubernetes cloud-native IDS data (redamorsli/kube-ids0): the two ML-ready files, one per application.\\n    Returns pools {'kube_dvwa': (X, y, fam), 'kube_boa': (...)} on the SHARED numeric columns, plus a report.\\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    files = {\\\"kube_dvwa\\\": glob.glob(os.path.join(path, \\\"**\\\", \\\"dvwa_dataset_ml_ready*.csv\\\"), recursive=True),\\n             \\\"kube_boa\\\": glob.glob(os.path.join(path, \\\"**\\\", \\\"boa_dataset_ml_ready*.csv\\\"), recursive=True)}\\n    frames, rep = {}, {}\\n    for k, fs in files.items():\\n        if not fs:\\n            continue\\n        df = pd.read_csv(fs[0], low_memory=False)\\n        lab = next((c for c in df.columns if \\\"label\\\" in c.lower() or c.lower() in (\\\"attack\\\", \\\"class\\\", \\\"target\\\")), None)\\n        rep[k] = {\\\"file\\\": os.path.basename(fs[0]), \\\"rows\\\": len(df), \\\"label_col\\\": lab, \\\"columns\\\": df.columns.tolist()[:60]}\\n        if lab is None:\\n            continue\\n        v = df[lab]\\n        fam = v.map(lambda x: KUBE_LABELS.get(int(x), \\\"other\\\")) if pd.api.types.is_numeric_dtype(v) else v.map(family_of)\\n        df = df.assign(__fam=fam.values)\\n        frames[k] = df\\n        if verbose:\\n            print(k, os.path.basename(fs[0]), df.shape, \\\"label:\\\", lab, \\\"families:\\\", df[\\\"__fam\\\"].value_counts().to_dict())\\n    if not frames:\\n        raise FileNotFoundError(\\\"no ml_ready kube files with a label column\\\")\\n    num = [set(c for c in f.columns if c != \\\"__fam\\\" and c.lower() not in LABEL_COLS and \\\"label\\\" not in c.lower()\\n               and not DROP_PATTERNS.search(c) and not re.search(r\\\"(ip|time|stamp|id$)\\\", c, re.I)\\n               and pd.api.types.is_numeric_dtype(f[c])) for f in frames.values()]\\n    common = sorted(set.intersection(*num)) if len(num) > 1 else sorted(num[0])\\n    if len(common) < 8 and len(frames) > 1:                     # the two apps share too little: keep DVWA only\\n        frames = {\\\"kube_dvwa\\\": frames[\\\"kube_dvwa\\\"]} if \\\"kube_dvwa\\\" in frames else dict(list(frames.items())[:1])\\n        common = sorted(num[0])\\n    rep[\\\"common_features\\\"] = common\\n    pools = {}\\n    for k, f in frames.items():\\n        X = f[common].astype(np.float32).replace([np.inf, -np.inf], np.nan).fillna(0).to_numpy()\\n        fam = f[\\\"__fam\\\"].to_numpy().astype(str)\\n        key = pd.util.hash_pandas_object(pd.concat([pd.DataFrame(X), pd.Series(fam)], axis=1), index=False).to_numpy()\\n        _, first = np.unique(key, return_index=True); first = np.sort(first)\\n        rep[k][\\\"dup_frac\\\"] = float(1 - len(first) / max(len(X), 1))\\n        X, fam = X[first], fam[first]\\n        if len(X) > max_rows:\\n            idx = rng.choice(len(X), max_rows, replace=False); X, fam = X[idx], fam[idx]\\n        nz = X.std(0) > 0\\n        pools[k] = (X, (fam != \\\"benign\\\").astype(np.int64), fam)\\n        rep[k][\\\"rows_final\\\"] = int(len(X))\\n    keep = np.ones(len(common), dtype=bool)\\n    for X, _, _ in pools.values():\\n        keep &= X.std(0) > 0\\n    pools = {k: (X[:, keep], y, f) for k, (X, y, f) in pools.items()}\\n    rep[\\\"n_features\\\"] = int(keep.sum())\\n    return pools, rep\\n\\n\\ndef synthetic_pools(n_features=43, n_sources=4, n_per=30000, seed=0, sep=0.22):\\n    \\\"\\\"\\\"Stand-in data for local smoke tests only. Never used for reported results.\\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    fams = [\\\"benign\\\", \\\"dos\\\", \\\"recon\\\", \\\"bruteforce\\\", \\\"web\\\", \\\"exploit\\\"]\\n    pools = {}\\n    centers = rng.normal(0, sep, size=(len(fams), n_features))\\n    for k in range(n_sources):\\n        shift = rng.normal(0, 0.15, size=n_features)\\n        mix = np.array([0.5, 0.2, 0.1, 0.07, 0.07, 0.06])\\n        fam = rng.choice(fams, size=n_per, p=mix)\\n        ci = np.array([fams.index(f) for f in fam])\\n        X = centers[ci] + shift + rng.normal(0, 1.0, size=(n_per, n_features))\\n        X = np.sign(X) * (np.expm1(np.abs(X) * 1.5))  # heavy tails like raw flow counters\\n        y = (fam != \\\"benign\\\").astype(np.int64)\\n        pools[f\\\"src{k}\\\"] = (X.astype(np.float32), y, fam.astype(str))\\n    return pools\\n\\n\\ndef report_json(reports: dict, path: str):\\n    with open(path, \\\"w\\\") as f:\\n        json.dump(reports, f, indent=2, default=str)\\n\", \"defenses.py\": \"\\\"\\\"\\\"Aggregation rules and contract-enforced acceptance rules.\\n\\nA Defense sees the matrix of submitted updates U [M, P] (after reveal) and\\nreturns per-update weights. Contract-style defenses also keep state across\\nrounds (reputation, bonds, burned probes).\\n\\nWhat an attacker may read before it submits is exposed by `public_view()`.\\nFor public rules this is the whole rule. For the sealed rule it is only the\\ncommitments and the declared check types, not the probe samples or the\\ndynamic thresholds.\\n\\nBaselines are our own re-implementations from the descriptions in the papers.\\nThey are named \\\"-style\\\" in the paper because the original code was not used.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nfrom dataclasses import dataclass, field\\n\\nimport numpy as np\\nimport torch\\n\\nfrom . import model as M\\n\\n\\ndef _cos(a: torch.Tensor, b: torch.Tensor) -> torch.Tensor:\\n    return (a @ b) / (a.norm(dim=-1) * b.norm() + 1e-12) if a.dim() == 2 else (a @ b) / (a.norm() * b.norm() + 1e-12)\\n\\n\\n@dataclass\\nclass Ctx:\\n    \\\"\\\"\\\"Round context passed to every defense.\\\"\\\"\\\"\\n    round: int\\n    theta: torch.Tensor          # current global model [P]\\n    shapes: list\\n    n_clients: int\\n    beacon: bytes = b\\\"\\\"\\n    sizes: torch.Tensor | None = None\\n\\n\\nclass Defense:\\n    name = \\\"base\\\"\\n    uses_state = False\\n\\n    def __init__(self, n_clients: int, device):\\n        self.n = n_clients\\n        self.device = device\\n        self.rep = torch.ones(n_clients, device=device)\\n        self.banned = torch.zeros(n_clients, dtype=torch.bool, device=device)\\n        self.bond = torch.ones(n_clients, device=device)\\n        self.log = {}\\n        self.last_w = None\\n\\n    # ---- interface ----\\n    def weights(self, U: torch.Tensor, ctx: Ctx) -> torch.Tensor:  # [M] non-negative\\n        raise NotImplementedError\\n\\n    def aggregate(self, U: torch.Tensor, ctx: Ctx) -> torch.Tensor:\\n        w = self.weights(U, ctx)\\n        w = w * (~self.banned).float()\\n        self.last_w = w.detach()\\n        if w.sum() <= 0:\\n            return torch.zeros(U.shape[1], device=U.device)\\n        return (w / w.sum()) @ U\\n\\n    def public_view(self) -> dict:\\n        return {}\\n\\n    def accepts(self, u: torch.Tensor, ref: torch.Tensor, ctx: Ctx, client: int) -> bool:\\n        \\\"\\\"\\\"Attacker-side dry run of the PUBLIC part of the rule. `ref` is the\\n        attacker's own estimate of the benign mean update.\\\"\\\"\\\"\\n        return True\\n\\n\\n# ------------------------------------------------------------------ plain aggregators\\nclass FedAvg(Defense):\\n    name = \\\"fedavg\\\"\\n\\n    def weights(self, U, ctx):\\n        return torch.ones(U.shape[0], device=U.device)\\n\\n\\nclass CoordMedian(Defense):\\n    name = \\\"median\\\"\\n\\n    def aggregate(self, U, ctx):\\n        self.last_w = None\\n        return U.median(0).values\\n\\n\\nclass TrimmedMean(Defense):\\n    name = \\\"trmean\\\"\\n\\n    def __init__(self, n_clients, device, beta=0.2):\\n        super().__init__(n_clients, device)\\n        self.beta = beta\\n\\n    def aggregate(self, U, ctx):\\n        self.last_w = None\\n        k = int(self.beta * U.shape[0])\\n        S = U.sort(0).values\\n        return S[k:U.shape[0] - k].mean(0) if k > 0 else S.mean(0)\\n\\n\\nclass MultiKrum(Defense):\\n    name = \\\"mkrum\\\"\\n\\n    def __init__(self, n_clients, device, f_frac=0.3, m=None):\\n        super().__init__(n_clients, device)\\n        self.f_frac = f_frac\\n        self.f = None\\n        self.m = m\\n\\n    def weights(self, U, ctx):\\n        n = U.shape[0]\\n        f = max(1, int(self.f_frac * n))\\n        k = max(1, n - f - 2)\\n        D = torch.cdist(U, U) ** 2\\n        sc = torch.stack([D[i][torch.arange(n, device=U.device) != i].sort().values[:k].sum() for i in range(n)])\\n        m = self.m or max(1, n - f)\\n        w = torch.zeros(n, device=U.device)\\n        w[sc.argsort()[:m]] = 1\\n        return w\\n\\n\\n# ------------------------------------------------------------------ contract rules (public)\\nclass NormGate(Defense):\\n    \\\"\\\"\\\"Norm-threshold contract (Nagarjun & Rajkumar style): reject if the update norm exceeds tau.\\n    tau is public: `tau_mult` x the running median honest norm, published each round.\\\"\\\"\\\"\\n    name = \\\"normgate\\\"\\n    uses_state = True\\n\\n    def __init__(self, n_clients, device, tau_mult=1.5):\\n        super().__init__(n_clients, device)\\n        self.tau_mult = tau_mult\\n        self.tau = float(\\\"inf\\\")\\n        self.ref_norm = None\\n\\n    def weights(self, U, ctx):\\n        norms = U.norm(dim=1)\\n        med = norms.median().item()\\n        self.ref_norm = med if self.ref_norm is None else 0.9 * self.ref_norm + 0.1 * med\\n        self.tau = self.tau_mult * self.ref_norm\\n        w = (norms <= self.tau).float()\\n        # reference norm uses only accepted updates next round (public state)\\n        if w.sum() > 0:\\n            self.ref_norm = float(norms[w > 0].median())\\n        return w\\n\\n    def public_view(self):\\n        return {\\\"tau\\\": self.tau, \\\"ref_norm\\\": self.ref_norm}\\n\\n    def accepts(self, u, ref, ctx, client):\\n        tau = self.tau_mult * (self.ref_norm if self.ref_norm is not None else ref.norm().item())\\n        return bool(u.norm() <= tau)\\n\\n\\nclass CosRep(Defense):\\n    \\\"\\\"\\\"PFAD-BC-style: reputation-weighted cosine aggregation (reconstructed from the abstract).\\n    Reference direction = coordinate median. Reputation is an EMA of clipped cosine; tenants\\n    below rho are dropped. Norms are clipped to the median norm.\\\"\\\"\\\"\\n    name = \\\"cosrep\\\"\\n    uses_state = True\\n\\n    def __init__(self, n_clients, device, lam=0.8, rho=0.3, cos_min=0.0):\\n        super().__init__(n_clients, device)\\n        self.lam, self.rho, self.cos_min = lam, rho, cos_min\\n        self.rep = torch.full((n_clients,), 0.5, device=device)\\n\\n    def weights(self, U, ctx):\\n        ref = U.median(0).values\\n        cs = torch.clamp(_cos(U, ref), min=0)\\n        self.rep = self.lam * self.rep + (1 - self.lam) * cs\\n        w = self.rep * (self.rep >= self.rho).float() * (cs > self.cos_min).float()\\n        norms = U.norm(dim=1)\\n        clip = torch.clamp(norms.median() / (norms + 1e-12), max=1.0)\\n        self._clip = clip\\n        return w * clip\\n\\n    def public_view(self):\\n        return {\\\"rep\\\": self.rep.clone(), \\\"rho\\\": self.rho, \\\"lam\\\": self.lam}\\n\\n    def accepts(self, u, ref, ctx, client):\\n        # public rule: cosine to the benign direction must stay > cos_min and reputation >= rho\\n        c = float(_cos(u, ref))\\n        return c > max(self.cos_min, 0.05)\\n\\n\\nclass TrustAware(Defense):\\n    \\\"\\\"\\\"Trust-aware aggregation (Houichi et al. style): per-round weights from distance to the\\n    weighted mean, smoothed into a trust score across rounds.\\\"\\\"\\\"\\n    name = \\\"trustaware\\\"\\n    uses_state = True\\n\\n    def __init__(self, n_clients, device, lam=0.7, temp=2.0):\\n        super().__init__(n_clients, device)\\n        self.lam, self.temp = lam, temp\\n        self.trust = torch.full((n_clients,), 0.5, device=device)\\n\\n    def weights(self, U, ctx):\\n        mu = (self.trust / self.trust.sum()) @ U\\n        d = (U - mu).norm(dim=1)\\n        s = torch.exp(-self.temp * d / (d.median() + 1e-12))\\n        self.trust = self.lam * self.trust + (1 - self.lam) * s\\n        return self.trust * (self.trust > 0.15).float()\\n\\n    def public_view(self):\\n        return {\\\"trust\\\": self.trust.clone(), \\\"lam\\\": self.lam}\\n\\n    def accepts(self, u, ref, ctx, client):\\n        return bool((u - ref).norm() <= 2.0 * ref.norm())\\n\\n\\n# ------------------------------------------------------------------ probe-based rules\\ndef probe_stats(theta_g: torch.Tensor, U: torch.Tensor, shapes, X: torch.Tensor, fam: np.ndarray, families):\\n    \\\"\\\"\\\"Per-family attack recall and benign false-alarm rate of theta_g and of every theta_g + u_i on a probe batch.\\n\\n    Returns R [C+1, F] (row 0 = current global model), fpr [C+1].\\\"\\\"\\\"\\n    cand = torch.cat([theta_g.unsqueeze(0), theta_g.unsqueeze(0) + U], 0)\\n    pred = M.predict(cand, shapes, X)\\n    dev = X.device\\n    ben = torch.as_tensor(fam == \\\"benign\\\", device=dev)\\n    fpr = (pred[:, ben] == 1).float().mean(1) if ben.any() else torch.zeros(cand.shape[0], device=dev)\\n    cols = []\\n    for f in families:\\n        if f == \\\"benign\\\":\\n            continue\\n        m = torch.as_tensor(fam == f, device=dev)\\n        cols.append((pred[:, m] == 1).float().mean(1) if m.any() else torch.ones(cand.shape[0], device=dev))\\n    return torch.stack(cols, 1), fpr\\n\\n\\ndef damage(theta_g: torch.Tensor, U: torch.Tensor, shapes, X: torch.Tensor, fam: np.ndarray, families):\\n    \\\"\\\"\\\"Worst-case harm of applying each update to the global model on a probe batch.\\n\\n    harm = max( largest drop in any attack-family recall , rise in benign false-alarm rate ).\\n    theta_g [P], U [C,P], X [N,in], fam [N] -> harm [C]\\n    \\\"\\\"\\\"\\n    R, fpr = probe_stats(theta_g, U, shapes, X, fam, families)\\n    drop = (R[0:1] - R[1:]).max(1).values\\n    return torch.maximum(drop, fpr[1:] - fpr[0]), None\\n\\n\\ndef peer_z(R: torch.Tensor, fpr: torch.Tensor, s_min: float, ref: tuple | None = None):\\n    \\\"\\\"\\\"Peer-relative badness. Each column is one check (attack-family recall, benign false alarms);\\n    an update is judged against the median and MAD of the OTHER submitted updates in that column.\\n\\n    R [M,F] recall, fpr [M]. Returns z [M,F+1]. `ref=(med, mad)` lets an attacker plug in its own estimate.\\\"\\\"\\\"\\n    S = torch.cat([-R, fpr.unsqueeze(1)], 1)              # larger = worse\\n    if ref is None:\\n        med = S.median(0).values\\n        mad = (S - med).abs().median(0).values\\n    else:\\n        med, mad = ref\\n    return (S - med) / torch.clamp(mad, min=s_min), (med, mad)\\n\\n\\nclass ProbeRule(Defense):\\n    \\\"\\\"\\\"Accept an update only if it does not damage detection on a probe batch.\\n\\n    Two independent design choices, kept separate so the experiments can tell which one matters:\\n\\n    public (probe source)\\n        True : one fixed probe set, readable by everyone (FLTrust-style root data on chain).\\n        False: SEALED. The pool is committed by Merkle root at setup. Each round a beacon picks fresh\\n               samples that were never used before (\\\"burn\\\"). They are revealed only after the\\n               submissions close, so nobody can fit to them in advance.\\n    mode (pass mark)\\n        \\\"abs\\\"    : fixed tolerance learned from clean calibration rounds and published on chain.\\n        \\\"rel_max\\\": the worst-case harm of an update must stay within median + kappa*MAD of the harm of\\n                   the other updates (unknown before reveal).\\n        \\\"rel_fam\\\": per-check peer test. Each family recall and the benign false-alarm rate is compared\\n                   with the median and MAD of the other updates for that same check; reject when any\\n                   check is more than kappa robust deviations worse (unknown before reveal).\\n    \\\"\\\"\\\"\\n    uses_state = True\\n\\n    def __init__(self, n_clients, device, pool, families, public=False, mode=\\\"rel_max\\\", probe_size=1024,\\n                 kappa=3.0, delta_floor=0.02, s_min=0.05, public_tol=0.15, calib_rounds=4, calib_q=0.95,\\n                 rep_lam=0.8, slash_margin=0.25, reuse=False, leak_frac=0.0, seed=0, use_rep=True):\\n        super().__init__(n_clients, device)\\n        self.public, self.mode = public, mode\\n        self.name = {(\\\"abs\\\", True): \\\"probe_public\\\", (\\\"abs\\\", False): \\\"sealed_abs\\\", (\\\"rel_max\\\", False): \\\"sealed\\\",\\n                     (\\\"rel_max\\\", True): \\\"public_rel\\\", (\\\"rel_fam\\\", True): \\\"public_fam\\\", (\\\"rel_fam\\\", False): \\\"sealed_fam\\\"}[(mode, public)]\\n        self.pool_X = torch.as_tensor(pool[\\\"X\\\"], device=device)\\n        self.pool_fam = np.asarray(pool[\\\"fam\\\"])\\n        self.families = [f for f in families if f != \\\"benign\\\"]\\n        self.probe_size = probe_size\\n        self.kappa, self.delta_floor, self.s_min, self.public_tol = kappa, delta_floor, s_min, public_tol\\n        self.public_thr = delta_floor + public_tol   # the only absolute pass mark an attacker can read\\n        self.calib_rounds, self.calib_q, self._calib = calib_rounds, calib_q, []\\n        self.rep_lam, self.slash_margin = rep_lam, slash_margin\\n        self.reuse, self.leak_frac, self.use_rep = reuse, leak_frac, use_rep\\n        self.rng = np.random.default_rng(seed)\\n        self.used = np.zeros(len(self.pool_fam), dtype=bool)\\n        self.rep = torch.full((n_clients,), 0.7, device=device)\\n        self.root = hashlib.sha256(np.ascontiguousarray(pool[\\\"X\\\"][:64]).tobytes() + str(len(pool[\\\"fam\\\"])).encode()).hexdigest()\\n        self.public_idx = self._draw(probe_size, burn=False) if public else None\\n        self.slashed = torch.zeros(n_clients, device=device)\\n        self.last = {}\\n\\n    # ---- probe drawing\\n    def _draw(self, n, burn=True, seed_bytes=b\\\"\\\"):\\n        avail = np.flatnonzero(~self.used) if (burn and not self.reuse) else np.arange(len(self.pool_fam))\\n        if len(avail) < n:  # pool exhausted: refresh (documented as a limitation)\\n            self.used[:] = False\\n            avail = np.arange(len(self.pool_fam))\\n        r = np.random.default_rng(int.from_bytes(hashlib.sha256(seed_bytes + str(self.rng.integers(1 << 30)).encode()).digest()[:8], \\\"little\\\"))\\n        fams = np.unique(self.pool_fam[avail])           # stratify so every family is checked every round\\n        per = max(1, n // len(fams))\\n        sel = []\\n        for f in fams:\\n            cand = avail[self.pool_fam[avail] == f]\\n            sel.append(r.choice(cand, min(per, len(cand)), replace=False))\\n        sel = np.concatenate(sel)\\n        if burn and not self.reuse:\\n            self.used[sel] = True\\n        return sel\\n\\n    def _probe(self, ctx):\\n        idx = self.public_idx if self.public else self._draw(self.probe_size, burn=True, seed_bytes=ctx.beacon)\\n        self.last_idx = idx\\n        return idx\\n\\n    def weights(self, U, ctx):\\n        idx = self._probe(ctx)\\n        Xp, famp = self.pool_X[idx], self.pool_fam[idx]\\n        R, fpr = probe_stats(ctx.theta, U, ctx.shapes, Xp, famp, self.families)\\n        if self.mode == \\\"rel_fam\\\":\\n            z, _ = peer_z(R[1:], fpr[1:], self.s_min)\\n            dmg = z.max(1).values\\n            thr = torch.as_tensor(self.kappa, device=U.device)\\n        else:\\n            dmg = torch.maximum((R[0:1] - R[1:]).max(1).values, fpr[1:] - fpr[0])\\n            if self.mode == \\\"abs\\\":\\n                if ctx.round < self.calib_rounds:   # clean calibration rounds: accept all, learn the honest spread\\n                    self._calib.append(dmg.detach().cpu().numpy())\\n                    if ctx.round == self.calib_rounds - 1:\\n                        self.public_thr = float(np.quantile(np.concatenate(self._calib), self.calib_q)) + self.delta_floor\\n                    thr = torch.as_tensor(float(\\\"inf\\\"), device=U.device)\\n                else:\\n                    thr = torch.as_tensor(self.public_thr, device=U.device)\\n            else:                                    # rel_max\\n                med = dmg.median()\\n                mad = (dmg - med).abs().median()\\n                thr = med + self.kappa * (mad + 1e-3) + self.delta_floor\\n        ok = (dmg <= thr).float()\\n        self.last = dict(dmg=dmg.detach().cpu().numpy(), thr=float(thr), ok=ok.detach().cpu().numpy())\\n        scale = float(thr) if np.isfinite(float(thr)) and float(thr) > 1e-6 else 1.0\\n        if self.use_rep:\\n            score = 1 - torch.clamp(dmg / scale, 0, 2) / 2\\n            self.rep = self.rep_lam * self.rep + (1 - self.rep_lam) * score\\n        # slash with attributable evidence (signed commitment + revealed update + revealed probes)\\n        margin = self.slash_margin * (scale if self.mode == \\\"rel_fam\\\" else 1.0)\\n        bad = dmg > (thr + margin)\\n        self.slashed += bad.float() * 0.5\\n        self.banned |= self.slashed >= 1.0\\n        return ok * (torch.clamp(self.rep, min=0.05) if self.use_rep else 1.0)\\n\\n    def public_view(self):\\n        v = {\\\"root\\\": self.root, \\\"probe_size\\\": self.probe_size, \\\"delta_floor\\\": self.delta_floor,\\n             \\\"kappa\\\": self.kappa, \\\"s_min\\\": self.s_min, \\\"mode\\\": self.mode, \\\"rep\\\": self.rep.clone()}\\n        if self.public:\\n            v[\\\"probe_idx\\\"] = self.public_idx\\n            v[\\\"pool_X\\\"], v[\\\"pool_fam\\\"] = self.pool_X, self.pool_fam\\n        elif self.leak_frac > 0:\\n            n = int(self.leak_frac * len(self.pool_fam))\\n            leak = np.random.default_rng(123).choice(len(self.pool_fam), n, replace=False)\\n            v[\\\"leak_idx\\\"] = leak\\n            v[\\\"pool_X\\\"], v[\\\"pool_fam\\\"] = self.pool_X, self.pool_fam\\n        return v\\n\\n    def accepts(self, u, ref, ctx, client):\\n        return True  # handled in the attacker via the probe it can actually see\\n\\n\\nPROBE_NAMES = {\\\"probe_public\\\": (True, \\\"abs\\\"), \\\"sealed_abs\\\": (False, \\\"abs\\\"), \\\"sealed\\\": (False, \\\"rel_max\\\"),\\n               \\\"public_rel\\\": (True, \\\"rel_max\\\"), \\\"public_fam\\\": (True, \\\"rel_fam\\\"), \\\"sealed_fam\\\": (False, \\\"rel_fam\\\")}\\n\\n\\ndef make_defense(name: str, n_clients: int, device, audit_pool=None, families=None, seed=0, **kw) -> Defense:\\n    if name == \\\"fedavg\\\":\\n        return FedAvg(n_clients, device)\\n    if name == \\\"median\\\":\\n        return CoordMedian(n_clients, device)\\n    if name == \\\"trmean\\\":\\n        return TrimmedMean(n_clients, device, **kw)\\n    if name == \\\"mkrum\\\":\\n        return MultiKrum(n_clients, device, **kw)\\n    if name == \\\"normgate\\\":\\n        return NormGate(n_clients, device, **kw)\\n    if name == \\\"cosrep\\\":\\n        return CosRep(n_clients, device, **kw)\\n    if name == \\\"trustaware\\\":\\n        return TrustAware(n_clients, device, **kw)\\n    if name in PROBE_NAMES:\\n        public, mode = PROBE_NAMES[name]\\n        return ProbeRule(n_clients, device, audit_pool, families, public=public, mode=mode, seed=seed, **kw)\\n    raise KeyError(name)\\n\", \"fl.py\": \"\\\"\\\"\\\"One federated run: tenants, a defense (contract rule), an optional attacker.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport json\\nimport time\\nfrom dataclasses import asdict, dataclass, field\\n\\nimport numpy as np\\nimport torch\\n\\nfrom . import model as M\\nfrom .attacks import Attacker\\nfrom .defenses import Ctx, make_defense, PROBE_NAMES\\nfrom .data import World, FAMILIES\\nfrom .accountability import Accountability\\n\\n\\n@dataclass\\nclass Config:\\n    defense: str = \\\"fedavg\\\"\\n    attack: str = \\\"none\\\"\\n    mal_frac: float = 0.3\\n    seed: int = 0\\n    rounds: int = 40\\n    local_epochs: int = 2\\n    bs: int = 128\\n    lr: float = 2e-3\\n    target: str = \\\"dos\\\"\\n    attack_start: int = 5\\n    farm_rounds: int = 10\\n    probe_size: int = 1024\\n    leak_frac: float = 0.0\\n    see_benign: bool = False         # True = no commit-reveal: attacker sees this round's benign updates\\n    reuse_probes: bool = False       # sealed rule: reuse revealed probes instead of burning them\\n    use_rep: bool = True\\n    hidden: tuple = (128, 64)\\n    eval_every: int = 5\\n    dparams: dict = field(default_factory=dict)   # defense hyper-parameters (tuned values go here)\\n    extra: dict = field(default_factory=dict)\\n\\n    def run_id(self):\\n        d = asdict(self)\\n        d.pop(\\\"extra\\\")\\n        d[\\\"dparams\\\"] = json.dumps(self.dparams, sort_keys=True)\\n        return hashlib.md5(repr(sorted(d.items())).encode()).hexdigest()[:12]\\n\\n\\ndef _stack(world: World, device):\\n    X = torch.as_tensor(np.stack([t[\\\"X\\\"] for t in world.tenants]), device=device)\\n    y = torch.as_tensor(np.stack([t[\\\"y\\\"] for t in world.tenants]), device=device)\\n    fam = np.stack([t[\\\"fam\\\"] for t in world.tenants])\\n    return X, y, fam\\n\\n\\ndef _eval(theta, shapes, world: World, target, device):\\n    \\\"\\\"\\\"Overall and per-source metrics, including AUC from attack-class probabilities.\\\"\\\"\\\"\\n    out = {}\\n    Ps, Ss, ys, fs = [], [], [], []\\n    for s, d in world.test.items():\\n        X = torch.as_tensor(d[\\\"X\\\"], device=device)\\n        pred = M.predict(theta, shapes, X)[0].cpu().numpy()\\n        sc = M.predict_proba(theta, shapes, X)[0].cpu().numpy()\\n        ms = M.metrics(pred, d[\\\"y\\\"], d[\\\"fam\\\"], target, score=sc)\\n        out[f\\\"f1_{s}\\\"] = ms[\\\"macro_f1\\\"]\\n        out[f\\\"rec_target_{s}\\\"] = ms.get(\\\"rec_target\\\", float(\\\"nan\\\"))\\n        out[f\\\"fpr_{s}\\\"] = ms[\\\"fpr\\\"]\\n        out[f\\\"auc_{s}\\\"] = ms.get(\\\"auc\\\", float(\\\"nan\\\"))\\n        for k_, v_ in ms.items():                               # per-family recall on this source (generalisation diagnostics)\\n            if k_.startswith(\\\"rec_\\\") and k_ != \\\"rec_target\\\":\\n                out[f\\\"{k_}_{s}\\\"] = v_\\n        Ps.append(pred); Ss.append(sc); ys.append(d[\\\"y\\\"]); fs.append(d[\\\"fam\\\"])\\n    out.update(M.metrics(np.concatenate(Ps), np.concatenate(ys), np.concatenate(fs), target, score=np.concatenate(Ss)))\\n    return out\\n\\n\\n@torch.no_grad()\\ndef mia_auc(theta, shapes, world: World, device, n: int = 3000, seed: int = 0) -> float:\\n    \\\"\\\"\\\"Loss-threshold membership inference: AUC of telling training rows (members) from test rows by the\\n    global model's loss. 0.5 = no leakage; higher = the model memorises its training tenants' data.\\\"\\\"\\\"\\n    import torch.nn.functional as F\\n    from sklearn.metrics import roc_auc_score\\n    rng = np.random.default_rng(seed)\\n    Xm = np.concatenate([t[\\\"X\\\"] for t in world.tenants]); ym = np.concatenate([t[\\\"y\\\"] for t in world.tenants])\\n    Xn = np.concatenate([d[\\\"X\\\"] for d in world.test.values()]); yn = np.concatenate([d[\\\"y\\\"] for d in world.test.values()])\\n    im = rng.choice(len(Xm), min(n, len(Xm)), replace=False); inn = rng.choice(len(Xn), min(n, len(Xn)), replace=False)\\n    def loss(X, y):\\n        X = torch.as_tensor(X, device=device); y = torch.as_tensor(y, device=device)\\n        lg = M.forward(theta.unsqueeze(0), shapes, X)[0]\\n        return F.cross_entropy(lg, y, reduction=\\\"none\\\").cpu().numpy()\\n    lm, ln = loss(Xm[im], ym[im]), loss(Xn[inn], yn[inn])\\n    return float(roc_auc_score(np.r_[np.ones(len(lm)), np.zeros(len(ln))], -np.r_[lm, ln]))\\n\\n\\ndef run_reference(world: World, cfg, device=\\\"cpu\\\", mode=\\\"centralized\\\", epochs: int = 20) -> dict:\\n    \\\"\\\"\\\"Reference points for the federated results.\\n    centralized: one model trained on all tenants' data pooled (upper bound, no privacy).\\n    local_only : every tenant trains alone on its own data; metrics averaged over tenants (lower bound).\\\"\\\"\\\"\\n    gen = torch.Generator(device=device).manual_seed(cfg.seed)\\n    shapes = M.layer_shapes(world.n_features, cfg.hidden, 2)\\n    theta = M.init_theta(shapes, cfg.seed, device)\\n    X, y, _ = _stack(world, device)\\n    if mode == \\\"centralized\\\":\\n        Xc = X.reshape(1, -1, X.shape[-1]); yc = y.reshape(1, -1)\\n        th = M.local_train(theta, shapes, Xc, yc, epochs, cfg.bs * 4, cfg.lr, gen)[0]\\n        final = _eval(th, shapes, world, cfg.target, device)\\n    else:\\n        th = M.local_train(theta, shapes, X, y, epochs, cfg.bs, cfg.lr, gen)\\n        evs = [_eval(th[i], shapes, world, cfg.target, device) for i in range(th.shape[0])]\\n        final = {k: float(np.nanmean([e[k] for e in evs])) for k in evs[0]}\\n    return dict(run_id=f\\\"{mode}-{cfg.seed}-{cfg.target}-{cfg.hidden}\\\", defense=mode, attack=\\\"none\\\", mal_frac=0.0,\\n                seed=cfg.seed, target=cfg.target, hidden=str(cfg.hidden), dparams=\\\"{}\\\", final=final, curve=[final])\\n\\n\\ndef pick_malicious(world: World, frac: float, target: str, rng) -> list:\\n    n = len(world.tenants)\\n    m = int(round(frac * n))\\n    if m == 0:\\n        return []\\n    share = np.array([np.mean(t[\\\"fam\\\"] == target) for t in world.tenants])\\n    elig = np.flatnonzero(share >= 0.05)             # attacker must actually hold target-family traffic\\n    if len(elig) < m:\\n        elig = np.argsort(-share)[: max(m, 1)]\\n    return sorted(rng.choice(elig, m, replace=False).tolist())\\n\\n\\ndef run_one(world: World, cfg: Config, device=\\\"cpu\\\", verbose=False, hooks=None) -> dict:\\n    \\\"\\\"\\\"hooks (optional): object with before(r, U_sub, mal) -> beacon bytes or None, and after(r, theta, defense, mal).\\n    Used by chain/integration.py to drive the on-chain round; experiments run with hooks=None.\\\"\\\"\\\"\\n    t0 = time.time()\\n    torch.manual_seed(cfg.seed)\\n    rng = np.random.default_rng(cfg.seed)\\n    gen = torch.Generator(device=device).manual_seed(cfg.seed)\\n    X, y, fam = _stack(world, device)\\n    C = X.shape[0]\\n    shapes = M.layer_shapes(world.n_features, cfg.hidden, 2)\\n    theta = M.init_theta(shapes, cfg.seed, device)\\n\\n    pool = {k: np.concatenate([world.audit[s][k] for s in world.audit]) for k in (\\\"X\\\", \\\"y\\\", \\\"fam\\\")}\\n    dkw = dict(cfg.dparams)\\n    if cfg.defense in PROBE_NAMES:\\n        dkw = {**{\\\"probe_size\\\": cfg.probe_size, \\\"leak_frac\\\": cfg.leak_frac, \\\"reuse\\\": cfg.reuse_probes,\\n                  \\\"use_rep\\\": cfg.use_rep}, **dkw}\\n    defense = make_defense(cfg.defense, C, device, audit_pool=pool, families=FAMILIES, seed=cfg.seed, **dkw)\\n    mal = pick_malicious(world, cfg.mal_frac, cfg.target, rng) if cfg.attack != \\\"none\\\" else []\\n    attacker = Attacker(cfg.attack, mal, C, farm_rounds=cfg.farm_rounds, see_benign=cfg.see_benign, seed=cfg.seed) if mal else None\\n    if attacker is not None:\\n        attacker.target = cfg.target\\n        attacker.train = dict(epochs=cfg.local_epochs, bs=cfg.bs, lr=cfg.lr, gen=gen)\\n        attacker.families = FAMILIES\\n        attacker.make_surrogate = lambda pool: make_defense(cfg.defense, C, device, audit_pool=pool, families=FAMILIES, seed=cfg.seed + 777, **dkw)\\n    y_poison = y.clone()\\n    tmask = torch.as_tensor(fam == cfg.target, device=device)\\n    y_poison[tmask] = 0                                    # target-family flows relabelled benign\\n    mal_t = torch.as_tensor(mal, device=device, dtype=torch.long) if mal else None\\n    acct = Accountability(C, alpha=cfg.extra.get('acct_alpha', 0.05), seed=cfg.seed)\\n    if cfg.extra.get(\\\"acct_trace\\\"):\\n        acct.trace = []\\n    beacon = b\\\"genesis\\\"\\n    curve, mass_mal, rej_hon, rej_mal = [], [], [], []\\n    honest_idx = [i for i in range(C) if i not in set(mal)]\\n    t_def = 0.0\\n\\n    for r in range(cfg.rounds):\\n        th_loc = M.local_train(theta, shapes, X, y, cfg.local_epochs, cfg.bs, cfg.lr, gen)\\n        U = th_loc - theta.unsqueeze(0)\\n        attacking = attacker is not None and r >= cfg.attack_start\\n        if attacking:\\n            th_p = M.local_train(theta, shapes, X[mal_t], y_poison[mal_t], cfg.local_epochs, cfg.bs, cfg.lr, gen)\\n            P = th_p - theta.unsqueeze(0)\\n            beacon = hashlib.sha256(beacon + str(r).encode()).digest()\\n            ctx = Ctx(r, theta, shapes, C, beacon)\\n            benign_U = U[torch.as_tensor(honest_idx, device=device)] if cfg.see_benign else None\\n            U_sub = U.clone()\\n            U_sub[mal_t] = attacker.craft(U, P, ctx, defense, world.tenants, benign_U)\\n        else:\\n            beacon = hashlib.sha256(beacon + str(r).encode()).digest()\\n            ctx = Ctx(r, theta, shapes, C, beacon)\\n            U_sub = U\\n        if hooks is not None:\\n            b = hooks.before(r, U_sub, mal)\\n            if b is not None:\\n                ctx = Ctx(r, theta, shapes, C, b)       # probes drawn from the ON-CHAIN beacon\\n        t1 = time.time()\\n        delta = defense.aggregate(U_sub, ctx)\\n        t_def += time.time() - t1\\n        theta = theta + delta\\n        acct.update(r, U_sub, defense)\\n        if hooks is not None:\\n            hooks.after(r, theta, defense, mal)\\n        w = defense.last_w\\n        if w is not None and w.sum() > 0 and mal:\\n            mass_mal.append(float(w[mal_t].sum() / w.sum()))\\n            rej_mal.append(float((w[mal_t] <= 0).float().mean()))\\n            rej_hon.append(float((w[torch.as_tensor(honest_idx, device=device)] <= 0).float().mean()))\\n        elif w is not None and w.sum() > 0:\\n            rej_hon.append(float((w <= 0).float().mean()))\\n        if (r + 1) % cfg.eval_every == 0 or r == cfg.rounds - 1:\\n            e = _eval(theta, shapes, world, cfg.target, device)\\n            e[\\\"round\\\"] = r + 1\\n            curve.append(e)\\n            if verbose:\\n                print(r + 1, {k: round(v, 3) for k, v in e.items() if k in (\\\"macro_f1\\\", \\\"rec_target\\\", \\\"fpr\\\")})\\n    final = curve[-1]\\n    row = dict(run_id=cfg.run_id(), **{k: v for k, v in asdict(cfg).items() if k not in (\\\"extra\\\", \\\"hidden\\\", \\\"dparams\\\")},\\n               dparams=json.dumps(cfg.dparams, sort_keys=True),\\n               n_mal=len(mal), final=final, curve=curve,\\n               mass_on_mal=float(np.mean(mass_mal)) if mass_mal else float(\\\"nan\\\"),\\n               mal_rejected=float(np.mean(rej_mal)) if rej_mal else float(\\\"nan\\\"),\\n               honest_rejected=float(np.mean(rej_hon)) if rej_hon else float(\\\"nan\\\"),\\n               defense_sec=t_def, wall_sec=time.time() - t0,\\n               banned=int(defense.banned.sum()), last_gamma=getattr(attacker, \\\"last_gamma\\\", float(\\\"nan\\\")),\\n               hidden=str(cfg.hidden), sec_per_round=(time.time() - t0) / max(cfg.rounds, 1),\\n               mia_auc=mia_auc(theta, shapes, world, device, seed=cfg.seed),\\n               **acct.summary(mal, cfg.attack_start))\\n    if acct.trace is not None:\\n        row[\\\"acct_trace\\\"] = dict(mal=list(map(int, mal)), rounds=acct.trace)\\n    return row\\n\", \"grid.py\": \"\\\"\\\"\\\"Experiment grids with resume. Each finished run is one JSON line, so a Kaggle session that dies\\ncan be restarted and continues where it stopped.\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport itertools\\nimport json\\nimport os\\nimport time\\n\\nfrom .fl import Config, run_one\\n\\nDEFENSES = [\\\"fedavg\\\", \\\"median\\\", \\\"trmean\\\", \\\"mkrum\\\", \\\"normgate\\\", \\\"cosrep\\\", \\\"trustaware\\\", \\\"probe_public\\\", \\\"sealed\\\"]\\nFINAL_DEFENSES = [\\\"fedavg\\\", \\\"median\\\", \\\"trmean\\\", \\\"mkrum\\\", \\\"normgate\\\", \\\"cosrep\\\", \\\"trustaware\\\",\\n                  \\\"probe_public\\\", \\\"sealed_abs\\\", \\\"sealed\\\", \\\"public_fam\\\", \\\"sealed_fam\\\"]\\nTUNE_SEEDS = (100, 101, 102)          # never used for the final numbers\\nFINAL_SEEDS = tuple(range(10))\\n\\n_Q = (0.9, 0.95, 0.99)\\nTUNE_GRIDS = {\\n    \\\"fedavg\\\": [{}], \\\"median\\\": [{}],\\n    \\\"trmean\\\": [{\\\"beta\\\": b} for b in (0.1, 0.2, 0.3, 0.4)],\\n    \\\"mkrum\\\": [{\\\"f_frac\\\": f} for f in (0.1, 0.2, 0.3)],\\n    \\\"normgate\\\": [{\\\"tau_mult\\\": t} for t in (1.0, 1.25, 1.5, 2.0, 3.0)],\\n    \\\"cosrep\\\": [{\\\"lam\\\": l, \\\"rho\\\": r} for l in (0.5, 0.8) for r in (0.1, 0.2, 0.3)],\\n    \\\"trustaware\\\": [{\\\"lam\\\": l, \\\"temp\\\": t} for l in (0.5, 0.7) for t in (1.0, 2.0, 4.0)],\\n    \\\"probe_public\\\": [{\\\"calib_q\\\": q, \\\"delta_floor\\\": f} for q in _Q for f in (0.0, 0.02, 0.05)],\\n    \\\"sealed_abs\\\": [{\\\"calib_q\\\": q, \\\"delta_floor\\\": f} for q in _Q for f in (0.0, 0.02, 0.05)],\\n    \\\"sealed\\\": [{\\\"kappa\\\": k, \\\"delta_floor\\\": f, \\\"use_rep\\\": r} for k in (2.0, 3.0, 5.0) for f in (0.02, 0.05, 0.1) for r in (True, False)],\\n    \\\"public_fam\\\": [{\\\"kappa\\\": k, \\\"s_min\\\": m, \\\"use_rep\\\": r} for k in (2.0, 3.0, 4.0, 6.0) for m in (0.02, 0.05, 0.1) for r in (True, False)],\\n    \\\"sealed_fam\\\": [{\\\"kappa\\\": k, \\\"s_min\\\": m, \\\"use_rep\\\": r} for k in (2.0, 3.0, 4.0, 6.0) for m in (0.02, 0.05, 0.1) for r in (True, False)],\\n}\\nTUNE_ATTACKS = [\\\"none\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\"]\\nATTACKS = [\\\"none\\\", \\\"targeted_lf\\\", \\\"signflip\\\", \\\"lie\\\", \\\"minmax\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\"]\\n\\n\\ndef preset(name: str, seeds=(0, 1, 2)):\\n    cfgs = []\\n    if name == \\\"gate\\\":          # day-7 go / no-go: does the rule-aware attack beat the public rules, does sealing stop it\\n        for d, a, s in itertools.product(DEFENSES, [\\\"none\\\", \\\"targeted_lf\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\"], seeds):\\n            cfgs.append(Config(defense=d, attack=a, mal_frac=0.3, seed=s))\\n    elif name == \\\"main\\\":\\n        for d, a, f, s in itertools.product(DEFENSES, ATTACKS, [0.1, 0.2, 0.3, 0.4], seeds):\\n            if a == \\\"none\\\" and f != 0.3:\\n                continue\\n            cfgs.append(Config(defense=d, attack=a, mal_frac=f, seed=s))\\n    elif name == \\\"ablation\\\":    # what each piece of the sealed rule buys\\n        for s in seeds:\\n            for kw in [dict(), dict(reuse_probes=True), dict(use_rep=False), dict(see_benign=True)]:\\n                for a in [\\\"rule_aware\\\", \\\"rule_aware_farm\\\"]:\\n                    cfgs.append(Config(defense=\\\"sealed\\\", attack=a, mal_frac=0.3, seed=s, **kw))\\n            for lk in [0.01, 0.05, 0.2, 0.5]:   # how much probe-pool leakage breaks sealing\\n                cfgs.append(Config(defense=\\\"sealed\\\", attack=\\\"rule_aware\\\", mal_frac=0.3, seed=s, leak_frac=lk))\\n    else:\\n        raise KeyError(name)\\n    return cfgs\\n\\n\\ndef tune_preset(seeds=TUNE_SEEDS, mal_frac=0.3):\\n    \\\"\\\"\\\"Same budget for every defense: each candidate setting is tried on the same attacks and seeds.\\\"\\\"\\\"\\n    cfgs = []\\n    for d, grid_ in TUNE_GRIDS.items():\\n        for dp in grid_:\\n            for a in TUNE_ATTACKS:\\n                for s in seeds:\\n                    cfgs.append(Config(defense=d, attack=a, mal_frac=mal_frac, seed=s, dparams=dp))\\n    return cfgs\\n\\n\\ndef final_preset(best: dict, seeds=FINAL_SEEDS, fracs=(0.1, 0.2, 0.3, 0.4),\\n                 attacks=(\\\"targeted_lf\\\", \\\"lie\\\", \\\"minmax\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\")):\\n    \\\"\\\"\\\"Final study: tuned settings, held-out seeds.\\\"\\\"\\\"\\n    cfgs = []\\n    for d in FINAL_DEFENSES:\\n        dp = best.get(d, {})\\n        for s in seeds:\\n            cfgs.append(Config(defense=d, attack=\\\"none\\\", mal_frac=0.3, seed=s, dparams=dp))\\n            for a in attacks:\\n                for f in fracs:\\n                    cfgs.append(Config(defense=d, attack=a, mal_frac=f, seed=s, dparams=dp))\\n    return cfgs\\n\\n\\nMEM_SEEDS = tuple(range(10, 20))        # fresh seeds, never used for tuning or for the first final study\\nMEM_DEFENSES = [\\\"fedavg\\\", \\\"median\\\", \\\"trmean\\\", \\\"cosrep\\\", \\\"probe_public\\\", \\\"sealed_abs\\\", \\\"public_fam\\\", \\\"sealed_fam\\\"]\\n\\n\\ndef memorize_preset(best: dict, seeds=MEM_SEEDS, fracs=(0.2, 0.3, 0.4)):\\n    \\\"\\\"\\\"POST-HOC follow-up (added after seeing the first final results): an attacker that memorizes the probe set it knows.\\n    Defense settings are the ones tuned on data half A; nothing is re-tuned.\\\"\\\"\\\"\\n    cfgs = []\\n    for d in MEM_DEFENSES:\\n        dp = best.get(d, {})\\n        for s in seeds:\\n            cfgs.append(Config(defense=d, attack=\\\"none\\\", mal_frac=0.3, seed=s, dparams=dp))\\n            for a in (\\\"rule_aware\\\", \\\"probe_memorize\\\"):\\n                for f in fracs:\\n                    cfgs.append(Config(defense=d, attack=a, mal_frac=f, seed=s, dparams=dp))\\n    for d in (\\\"sealed_fam\\\", \\\"sealed_abs\\\"):                    # how much probe-pool leakage breaks sealing\\n        for s in seeds:\\n            for lk in (0.01, 0.05, 0.2, 0.5):\\n                cfgs.append(Config(defense=d, attack=\\\"probe_memorize\\\", mal_frac=0.3, seed=s, dparams=best.get(d, {}), leak_frac=lk))\\n    return cfgs\\n\\n\\nSETTLE_SEEDS = tuple(range(20, 50))      # 30 fresh seeds, confirmatory\\nSTRONG_SEEDS = tuple(range(50, 60))      # 10 fresh seeds for the optimising attacker\\nSETTLE_DEFENSES = [\\\"probe_public\\\", \\\"public_fam\\\", \\\"sealed_abs\\\", \\\"sealed_fam\\\"]\\nSTRONG_DEFENSES = [\\\"fedavg\\\", \\\"median\\\", \\\"trmean\\\", \\\"mkrum\\\", \\\"normgate\\\", \\\"cosrep\\\", \\\"trustaware\\\",\\n                   \\\"probe_public\\\", \\\"sealed_abs\\\", \\\"public_fam\\\", \\\"sealed_fam\\\"]\\n\\n\\ndef settle_preset(best: dict, seeds=SETTLE_SEEDS):\\n    \\\"\\\"\\\"Confirmatory: sealed vs public probes under the memorizing attacker. Primary cell = probe_memorize at 30%.\\\"\\\"\\\"\\n    cfgs = []\\n    for d in SETTLE_DEFENSES:\\n        dp = best.get(d, {})\\n        for s in seeds:\\n            cfgs.append(Config(defense=d, attack=\\\"none\\\", mal_frac=0.3, seed=s, dparams=dp))\\n            cfgs.append(Config(defense=d, attack=\\\"rule_aware\\\", mal_frac=0.3, seed=s, dparams=dp))\\n            for f in (0.2, 0.3, 0.4):\\n                cfgs.append(Config(defense=d, attack=\\\"probe_memorize\\\", mal_frac=f, seed=s, dparams=dp))\\n    return cfgs\\n\\n\\ndef strong_preset(best: dict, seeds=STRONG_SEEDS):\\n    \\\"\\\"\\\"Defense-agnostic optimising attacker against every rule, with the older attackers on the same seeds for pairing.\\\"\\\"\\\"\\n    cfgs = []\\n    for d in STRONG_DEFENSES:\\n        dp = best.get(d, {})\\n        for s in seeds:\\n            cfgs.append(Config(defense=d, attack=\\\"none\\\", mal_frac=0.3, seed=s, dparams=dp))\\n            for a in (\\\"rule_aware\\\", \\\"probe_memorize\\\"):\\n                cfgs.append(Config(defense=d, attack=a, mal_frac=0.3, seed=s, dparams=dp))\\n            for f in (0.3, 0.4):\\n                cfgs.append(Config(defense=d, attack=\\\"opt_adaptive\\\", mal_frac=f, seed=s, dparams=dp))\\n    return cfgs\\n\\n\\n# ------------------------------------------------------------------ Phase 2 (robustness checks, fresh seeds, tuned settings)\\nP2_DEFENSES = [\\\"fedavg\\\", \\\"median\\\", \\\"trmean\\\", \\\"cosrep\\\", \\\"probe_public\\\", \\\"sealed_fam\\\"]\\nACCT_DEFENSES = [\\\"probe_public\\\", \\\"sealed_abs\\\", \\\"public_fam\\\", \\\"sealed_fam\\\", \\\"cosrep\\\", \\\"median\\\"]\\nACCT_ATTACKS = [\\\"none\\\", \\\"rule_aware\\\", \\\"rule_aware_farm\\\", \\\"probe_memorize\\\", \\\"opt_adaptive\\\"]\\n\\n\\ndef p2_preset(best: dict, defenses, attacks, seeds, fracs=(0.3,), **kw):\\n    cfgs = []\\n    for d in defenses:\\n        dp = best.get(d, {})\\n        for s in seeds:\\n            for a in attacks:\\n                for f in (fracs if a != \\\"none\\\" else (0.3,)):\\n                    cfgs.append(Config(defense=d, attack=a, mal_frac=f, seed=s, dparams=dp, **kw))\\n    return cfgs\\n\\n\\ndef ablation_preset(best: dict, seeds=FINAL_SEEDS):\\n    \\\"\\\"\\\"What each mechanism buys, and how much leakage or a missing commit-reveal step costs.\\\"\\\"\\\"\\n    cfgs = []\\n    for s in seeds:\\n        for a in (\\\"rule_aware\\\", \\\"rule_aware_farm\\\"):\\n            for d in (\\\"sealed_fam\\\", \\\"sealed\\\"):\\n                dp = best.get(d, {})\\n                for kw in (dict(reuse_probes=True), dict(see_benign=True), dict(use_rep=False)):\\n                    cfgs.append(Config(defense=d, attack=a, mal_frac=0.3, seed=s, dparams={k: v for k, v in dp.items() if k != \\\"use_rep\\\"} if \\\"use_rep\\\" in kw else dp, **kw))\\n            for lk in (0.01, 0.05, 0.2, 0.5):                 # probe-pool leakage to the attacker\\n                cfgs.append(Config(defense=\\\"sealed_fam\\\", attack=a, mal_frac=0.3, seed=s, dparams=best.get(\\\"sealed_fam\\\", {}), leak_frac=lk))\\n    return cfgs\\n\\n\\ndef run_grid(world, cfgs, out_path, device=\\\"cuda\\\", tag=\\\"\\\"):\\n    done = set()\\n    if os.path.exists(out_path):\\n        for line in open(out_path):\\n            try:\\n                done.add(json.loads(line)[\\\"run_id\\\"])\\n            except Exception:\\n                pass\\n    todo = [c for c in cfgs if c.run_id() not in done]\\n    print(f\\\"{len(cfgs)} configs, {len(done)} done, {len(todo)} to run\\\")\\n    t0 = time.time()\\n    with open(out_path, \\\"a\\\") as f:\\n        for i, c in enumerate(todo):\\n            try:\\n                row = run_one(world, c, device)\\n            except Exception as e:  # keep the sweep alive, record the failure\\n                row = dict(run_id=c.run_id(), error=repr(e), defense=c.defense, attack=c.attack, seed=c.seed)\\n            row[\\\"tag\\\"] = tag\\n            f.write(json.dumps(row, default=float) + \\\"\\\\n\\\")\\n            f.flush()\\n            if i % 10 == 0:\\n                el = time.time() - t0\\n                print(f\\\"[{i+1}/{len(todo)}] {c.defense:12s} {c.attack:16s} mal={c.mal_frac} seed={c.seed}  {el/60:.1f} min elapsed\\\", flush=True)\\n\\n\\ndef _worker(world, cfgs, out_path, device, tag):\\n    run_grid(world, cfgs, out_path, device=device, tag=tag)\\n\\n\\ndef run_parallel(world, cfgs, out_prefix, devices, tag=\\\"\\\"):\\n    \\\"\\\"\\\"One process per GPU. Each process writes its own JSONL shard, so there are no write clashes.\\\"\\\"\\\"\\n    import multiprocessing as mp\\n\\n    ctx = mp.get_context(\\\"spawn\\\")\\n    procs = []\\n    for i, dev in enumerate(devices):\\n        shard = cfgs[i::len(devices)]\\n        p = ctx.Process(target=_worker, args=(world, shard, f\\\"{out_prefix}.shard{i}.jsonl\\\", dev, tag))\\n        p.start()\\n        procs.append(p)\\n    for p in procs:\\n        p.join()\\n\\n\\n# ------------------------------------------------------------------ R3 (pre-registered in docs/PREREG_R3.md): how far the attack result reaches\\nR3_SEEDS = dict(H1B=tuple(range(410, 420)), MAIN=tuple(range(350, 360)), ARCH=tuple(range(420, 425)), BUDGET8=tuple(range(350, 355)))\\nR3_BUDGET8_DEFENSES = [\\\"median\\\", \\\"trustaware\\\", \\\"cosrep\\\", \\\"sealed_fam\\\", \\\"public_fam\\\"]\\n\\n\\ndef r3_preset(best: dict, stage: str):\\n    S, D_ = R3_SEEDS, STRONG_DEFENSES\\n    if stage == \\\"H1B\\\":        # 10 more seeds for the primary test (clean + optimiser, 30%)\\n        return p2_preset(best, D_, [\\\"none\\\", \\\"opt_adaptive\\\"], S[\\\"H1B\\\"], fracs=(0.3,))\\n    if stage == \\\"ES\\\":         # independent search procedure, same budget\\n        return p2_preset(best, D_, [\\\"opt_es\\\"], S[\\\"MAIN\\\"], fracs=(0.3,))\\n    if stage == \\\"AGR\\\":        # published AGR-tailored attack against the simulated rule\\n        return p2_preset(best, D_, [\\\"agr_tailored\\\"], S[\\\"MAIN\\\"], fracs=(0.3,))\\n    if stage == \\\"FRAC\\\":       # fewer malicious tenants\\n        return p2_preset(best, D_, [\\\"opt_adaptive\\\"], S[\\\"MAIN\\\"], fracs=(0.1, 0.2))\\n    if stage == \\\"KNOW\\\":       # imperfect knowledge: noisier honest-update estimates; one tenth of own data\\n        return p2_preset(best, D_, [\\\"opt_noisy\\\", \\\"opt_lowdata\\\"], S[\\\"MAIN\\\"], fracs=(0.3,))\\n    if stage == \\\"BUDGET2\\\":\\n        return p2_preset(best, D_, [\\\"opt_adaptive_x2\\\"], S[\\\"MAIN\\\"], fracs=(0.3,))\\n    if stage == \\\"ARCH\\\":       # different model and local training: 256-128-64 MLP, 3 local epochs, lr 1e-3\\n        return p2_preset(best, P2_DEFENSES, [\\\"none\\\", \\\"opt_adaptive\\\"], S[\\\"ARCH\\\"], fracs=(0.3,), hidden=(256, 128, 64), local_epochs=3, lr=1e-3)\\n    if stage == \\\"BUDGET8\\\":\\n        return p2_preset(best, R3_BUDGET8_DEFENSES, [\\\"opt_adaptive_x8\\\"], S[\\\"BUDGET8\\\"], fracs=(0.3,))\\n    raise KeyError(stage)\\n\", \"model.py\": \"\\\"\\\"\\\"Batched MLP so many tenants train at once on one GPU.\\n\\nEach tenant has its own weights, stored with a leading \\\"client\\\" dimension and\\nmultiplied with bmm. Adam is element-wise, so training C clients together is\\nmathematically identical to training them one by one.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport math\\n\\nimport numpy as np\\nimport torch\\nimport torch.nn.functional as F\\n\\n\\ndef layer_shapes(n_in: int, hidden=(128, 64), n_out: int = 2):\\n    dims = [n_in, *hidden, n_out]\\n    return [(dims[i], dims[i + 1]) for i in range(len(dims) - 1)]\\n\\n\\ndef n_params(shapes) -> int:\\n    return sum(i * o + o for i, o in shapes)\\n\\n\\ndef init_theta(shapes, seed: int, device) -> torch.Tensor:\\n    g = torch.Generator().manual_seed(seed)\\n    parts = []\\n    for i, o in shapes:\\n        bound = 1 / math.sqrt(i)\\n        parts.append(((torch.rand(i * o, generator=g) * 2 - 1) * bound))\\n        parts.append(torch.zeros(o))\\n    return torch.cat(parts).to(device)\\n\\n\\ndef unflatten(theta: torch.Tensor, shapes):\\n    \\\"\\\"\\\"theta: [C, P] -> list of (W [C,i,o], b [C,1,o]).\\\"\\\"\\\"\\n    C = theta.shape[0]\\n    out, k = [], 0\\n    for i, o in shapes:\\n        W = theta[:, k:k + i * o].reshape(C, i, o)\\n        k += i * o\\n        b = theta[:, k:k + o].reshape(C, 1, o)\\n        k += o\\n        out.append((W, b))\\n    return out\\n\\n\\ndef forward(theta: torch.Tensor, shapes, x: torch.Tensor) -> torch.Tensor:\\n    \\\"\\\"\\\"theta [C,P], x [C,B,in] (or [B,in] shared) -> logits [C,B,out].\\\"\\\"\\\"\\n    layers = unflatten(theta, shapes)\\n    h = x if x.dim() == 3 else x.unsqueeze(0).expand(theta.shape[0], -1, -1)\\n    for n, (W, b) in enumerate(layers):\\n        h = torch.baddbmm(b, h, W)\\n        if n < len(layers) - 1:\\n            h = F.relu(h)\\n    return h\\n\\n\\n@torch.no_grad()\\ndef predict(theta: torch.Tensor, shapes, X: torch.Tensor, bs: int = 8192) -> torch.Tensor:\\n    \\\"\\\"\\\"theta [C,P] (or [P]) on shared X [N,in] -> predicted class [C,N].\\\"\\\"\\\"\\n    if theta.dim() == 1:\\n        theta = theta.unsqueeze(0)\\n    outs = []\\n    for i in range(0, X.shape[0], bs):\\n        outs.append(forward(theta, shapes, X[i:i + bs]).argmax(-1))\\n    return torch.cat(outs, 1)\\n\\n\\n@torch.no_grad()\\ndef predict_proba(theta: torch.Tensor, shapes, X: torch.Tensor, bs: int = 8192) -> torch.Tensor:\\n    \\\"\\\"\\\"Probability of the attack class, [C,N].\\\"\\\"\\\"\\n    if theta.dim() == 1:\\n        theta = theta.unsqueeze(0)\\n    outs = [torch.softmax(forward(theta, shapes, X[i:i + bs]), -1)[..., 1] for i in range(0, X.shape[0], bs)]\\n    return torch.cat(outs, 1)\\n\\n\\ndef local_train(theta0: torch.Tensor, shapes, X: torch.Tensor, y: torch.Tensor, epochs: int,\\n                bs: int, lr: float, gen: torch.Generator, w_pos: float = 1.0) -> torch.Tensor:\\n    \\\"\\\"\\\"Train C clients in parallel from the same start.\\n\\n    theta0 [P]; X [C,N,in]; y [C,N]. Returns the C trained parameter vectors [C,P].\\n    \\\"\\\"\\\"\\n    C, N, _ = X.shape\\n    th = theta0.unsqueeze(0).repeat(C, 1).clone().requires_grad_(True)\\n    opt = torch.optim.Adam([th], lr=lr)\\n    wt = torch.tensor([1.0, w_pos], device=X.device)\\n    steps = math.ceil(N / bs)\\n    ar = torch.arange(C, device=X.device).unsqueeze(1)\\n    for _ in range(epochs):\\n        perm = torch.argsort(torch.rand(C, N, generator=gen, device=X.device), dim=1)\\n        for s in range(steps):\\n            idx = perm[:, s * bs:(s + 1) * bs]\\n            xb, yb = X[ar, idx], y[ar, idx]\\n            logit = forward(th, shapes, xb)\\n            loss = F.cross_entropy(logit.reshape(-1, logit.shape[-1]), yb.reshape(-1), weight=wt, reduction=\\\"none\\\")\\n            loss = loss.reshape(C, -1).mean(1).sum()  # sum over clients keeps them independent\\n            opt.zero_grad(set_to_none=True)\\n            loss.backward()\\n            opt.step()\\n    return th.detach()\\n\\n\\ndef metrics(pred: np.ndarray, y: np.ndarray, fam: np.ndarray, target: str = \\\"dos\\\", score=None) -> dict:\\n    \\\"\\\"\\\"Binary IDS metrics plus per-family recall (fraction of attack flows flagged).\\\"\\\"\\\"\\n    tp = ((pred == 1) & (y == 1)).sum()\\n    fp = ((pred == 1) & (y == 0)).sum()\\n    fn = ((pred == 0) & (y == 1)).sum()\\n    tn = ((pred == 0) & (y == 0)).sum()\\n    prec = tp / max(tp + fp, 1)\\n    rec = tp / max(tp + fn, 1)\\n    f1_att = 2 * prec * rec / max(prec + rec, 1e-9)\\n    prec_b = tn / max(tn + fn, 1)\\n    rec_b = tn / max(tn + fp, 1)\\n    f1_ben = 2 * prec_b * rec_b / max(prec_b + rec_b, 1e-9)\\n    out = dict(acc=float((tp + tn) / max(len(y), 1)), macro_f1=float((f1_att + f1_ben) / 2),\\n               fpr=float(fp / max(fp + tn, 1)), recall=float(rec))\\n    for f in np.unique(fam):\\n        if f == \\\"benign\\\":\\n            continue\\n        m = fam == f\\n        out[f\\\"rec_{f}\\\"] = float((pred[m] == 1).mean()) if m.any() else float(\\\"nan\\\")\\n    out[\\\"rec_target\\\"] = out.get(f\\\"rec_{target}\\\", float(\\\"nan\\\"))\\n    out[\\\"precision\\\"] = float(prec)\\n    out[\\\"f1_attack\\\"] = float(f1_att)\\n    if score is not None and len(np.unique(y)) == 2:\\n        from sklearn.metrics import roc_auc_score\\n        out[\\\"auc\\\"] = float(roc_auc_score(y, score))\\n    return out\\n\", \"pqc.py\": \"\\\"\\\"\\\"Post-quantum layer: ML-DSA signed update commitments and ML-KEM protected transport.\\n\\nWhat is real here (measured, not estimated):\\n  * ML-DSA-{44,65,87} sign / verify and ML-KEM-{512,768,1024} keygen / encapsulate / decapsulate\\n    through liboqs, plus sizes.\\n  * A hybrid envelope: ML-KEM shared secret -> HKDF-SHA256 -> AES-256-GCM over the update bytes.\\n  * Calldata gas for posting a signature on an EVM chain, computed exactly from EIP-2028\\n    (4 gas per zero byte, 16 per non-zero byte) on the real signature bytes.\\nWhat is NOT done here: verifying ML-DSA inside the EVM. That cost is not measured by this module.\\nThe contract design verifies off-chain and records a hash on-chain with a challenge path.\\n\\\"\\\"\\\"\\nfrom __future__ import annotations\\n\\nimport hashlib\\nimport os\\nimport struct\\nimport time\\nfrom dataclasses import dataclass\\n\\nimport numpy as np\\n\\ntry:\\n    import oqs\\n    HAVE_OQS = True\\nexcept Exception:  # pragma: no cover\\n    HAVE_OQS = False\\n\\nfrom cryptography.hazmat.primitives import hashes\\nfrom cryptography.hazmat.primitives.asymmetric.ed25519 import Ed25519PrivateKey\\nfrom cryptography.hazmat.primitives.ciphers.aead import AESGCM\\nfrom cryptography.hazmat.primitives.kdf.hkdf import HKDF\\nfrom cryptography.exceptions import InvalidSignature\\n\\n\\ndef calldata_gas(b: bytes) -> int:\\n    z = b.count(0)\\n    return 4 * z + 16 * (len(b) - z)\\n\\n\\ndef update_bytes(delta: np.ndarray) -> bytes:\\n    return np.asarray(delta, dtype=np.float32).tobytes()\\n\\n\\ndef commitment(round_: int, tenant: int, delta: np.ndarray, salt: bytes) -> bytes:\\n    \\\"\\\"\\\"What gets signed: binds the update to a round, a tenant and a fresh salt (replay protection).\\\"\\\"\\\"\\n    return hashlib.sha256(struct.pack(\\\">II\\\", round_, tenant) + salt + hashlib.sha256(update_bytes(delta)).digest()).digest()\\n\\n\\nclass MLDSA:\\n    def __init__(self, alg=\\\"ML-DSA-65\\\"):\\n        self.alg = alg\\n        self.sig = oqs.Signature(alg)\\n        self.pk = self.sig.generate_keypair()\\n\\n    def sign(self, msg: bytes) -> bytes:\\n        return self.sig.sign(msg)\\n\\n    @staticmethod\\n    def verify(alg, msg, sig, pk) -> bool:\\n        with oqs.Signature(alg) as v:\\n            return bool(v.verify(msg, sig, pk))\\n\\n\\nclass Ed25519:\\n    def __init__(self):\\n        self.sk = Ed25519PrivateKey.generate()\\n        self.pk = self.sk.public_key()\\n\\n    def sign(self, msg):\\n        return self.sk.sign(msg)\\n\\n    def verify(self, msg, sig) -> bool:\\n        try:\\n            self.pk.verify(sig, msg)\\n            return True\\n        except InvalidSignature:\\n            return False\\n\\n\\ndef kem_envelope(kem_alg: str, recipient_pk: bytes, payload: bytes):\\n    \\\"\\\"\\\"Sender side: encapsulate to the recipient's ML-KEM public key, encrypt payload with AES-GCM.\\\"\\\"\\\"\\n    with oqs.KeyEncapsulation(kem_alg) as k:\\n        ct, ss = k.encap_secret(recipient_pk)\\n    key = HKDF(hashes.SHA256(), 32, None, b\\\"qbfl-update-v1\\\").derive(ss)\\n    nonce = os.urandom(12)\\n    return ct, nonce, AESGCM(key).encrypt(nonce, payload, b\\\"\\\")\\n\\n\\ndef kem_open(kem: \\\"oqs.KeyEncapsulation\\\", ct, nonce, blob) -> bytes:\\n    ss = kem.decap_secret(ct)\\n    key = HKDF(hashes.SHA256(), 32, None, b\\\"qbfl-update-v1\\\").derive(ss)\\n    return AESGCM(key).decrypt(nonce, blob, b\\\"\\\")\\n\\n\\ndef bench(n_params: int, n_tenants: int, sig_alg=\\\"ML-DSA-65\\\", kem_alg=\\\"ML-KEM-768\\\", reps=50, seed=0) -> dict:\\n    \\\"\\\"\\\"Per-round cost of signing + verifying + sealing K tenant updates, measured.\\\"\\\"\\\"\\n    rng = np.random.default_rng(seed)\\n    delta = rng.normal(size=n_params).astype(np.float32)\\n    out = {\\\"n_params\\\": n_params, \\\"update_bytes\\\": n_params * 4, \\\"n_tenants\\\": n_tenants}\\n    s = MLDSA(sig_alg)\\n    ed = Ed25519()\\n    msg = commitment(1, 0, delta, os.urandom(16))\\n    ts = {k: [] for k in (\\\"pq_sign\\\", \\\"pq_verify\\\", \\\"ed_sign\\\", \\\"ed_verify\\\", \\\"kem_keygen\\\", \\\"kem_encap\\\", \\\"kem_decap\\\")}\\n    for _ in range(reps):\\n        t = time.perf_counter(); sg = s.sign(msg); ts[\\\"pq_sign\\\"].append(time.perf_counter() - t)\\n        t = time.perf_counter(); ok = MLDSA.verify(sig_alg, msg, sg, s.pk); ts[\\\"pq_verify\\\"].append(time.perf_counter() - t)\\n        assert ok\\n        t = time.perf_counter(); eg = ed.sign(msg); ts[\\\"ed_sign\\\"].append(time.perf_counter() - t)\\n        t = time.perf_counter(); assert ed.verify(msg, eg); ts[\\\"ed_verify\\\"].append(time.perf_counter() - t)\\n    with oqs.KeyEncapsulation(kem_alg) as kem:\\n        t = time.perf_counter()\\n        for _ in range(reps):\\n            pk = kem.generate_keypair()\\n        ts[\\\"kem_keygen\\\"] = [(time.perf_counter() - t) / reps]\\n        pt = update_bytes(delta)\\n        for _ in range(reps):\\n            t = time.perf_counter(); ct, nonce, blob = kem_envelope(kem_alg, pk, pt); e = time.perf_counter() - t\\n            ts[\\\"kem_encap\\\"].append(e)\\n            t = time.perf_counter(); back = kem_open(kem, ct, nonce, blob); ts[\\\"kem_decap\\\"].append(time.perf_counter() - t)\\n            assert back == pt\\n    for k, v in ts.items():\\n        out[f\\\"{k}_ms\\\"] = float(np.median(v) * 1e3)\\n    out.update(pq_pk_bytes=len(s.pk), pq_sig_bytes=len(sg), ed_sig_bytes=len(eg), kem_pk_bytes=len(pk), kem_ct_bytes=len(ct),\\n               envelope_overhead_bytes=len(ct) + 12 + 16,\\n               pq_sig_calldata_gas=calldata_gas(sg), ed_sig_calldata_gas=calldata_gas(eg),\\n               round_sign_verify_ms=n_tenants * (out[\\\"pq_sign_ms\\\"] + out[\\\"pq_verify_ms\\\"]),\\n               round_seal_open_ms=n_tenants * (out[\\\"kem_encap_ms\\\"] + out[\\\"kem_decap_ms\\\"]),\\n               round_calldata_gas=n_tenants * (calldata_gas(sg) + 32))\\n    return out\\n\\n\\ndef security_checks(sig_alg=\\\"ML-DSA-65\\\", kem_alg=\\\"ML-KEM-768\\\", n_params=2000) -> dict:\\n    \\\"\\\"\\\"Properties the protocol relies on, each tested by an attempted attack. Returns pass/fail.\\\"\\\"\\\"\\n    rng = np.random.default_rng(1)\\n    delta = rng.normal(size=n_params).astype(np.float32)\\n    a, b = MLDSA(sig_alg), MLDSA(sig_alg)\\n    salt = os.urandom(16)\\n    msg = commitment(7, 3, delta, salt)\\n    sg = a.sign(msg)\\n    res = {}\\n    res[\\\"valid_signature_accepts\\\"] = MLDSA.verify(sig_alg, msg, sg, a.pk)\\n    tam = delta.copy(); tam[0] += 1e-3\\n    res[\\\"tampered_update_rejected\\\"] = not MLDSA.verify(sig_alg, commitment(7, 3, tam, salt), sg, a.pk)\\n    res[\\\"wrong_round_rejected\\\"] = not MLDSA.verify(sig_alg, commitment(8, 3, delta, salt), sg, a.pk)\\n    res[\\\"wrong_tenant_rejected\\\"] = not MLDSA.verify(sig_alg, commitment(7, 4, delta, salt), sg, a.pk)\\n    res[\\\"other_key_rejected\\\"] = not MLDSA.verify(sig_alg, msg, sg, b.pk)\\n    bad = bytearray(sg); bad[10] ^= 1\\n    res[\\\"bitflipped_signature_rejected\\\"] = not MLDSA.verify(sig_alg, msg, bytes(bad), a.pk)\\n    with oqs.KeyEncapsulation(kem_alg) as kem:\\n        pk = kem.generate_keypair()\\n        ct, nonce, blob = kem_envelope(kem_alg, pk, update_bytes(delta))\\n        res[\\\"envelope_roundtrip\\\"] = kem_open(kem, ct, nonce, blob) == update_bytes(delta)\\n        tb = bytearray(blob); tb[5] ^= 1\\n        try:\\n            kem_open(kem, ct, nonce, bytes(tb)); res[\\\"tampered_ciphertext_rejected\\\"] = False\\n        except Exception:\\n            res[\\\"tampered_ciphertext_rejected\\\"] = True\\n    return res\\n\\n\\nclass Ledger:\\n    \\\"\\\"\\\"Minimal accountable ledger used by the experiments: records signed commitments and lets anyone\\n    check a dispute (challenge) by re-verifying the signature over the revealed update.\\\"\\\"\\\"\\n\\n    def __init__(self, sig_alg=\\\"ML-DSA-65\\\"):\\n        self.alg = sig_alg\\n        self.keys, self.entries, self.bond = {}, {}, {}\\n\\n    def register(self, tenant, pk, bond=1.0):\\n        self.keys[tenant] = pk\\n        self.bond[tenant] = bond\\n\\n    def post(self, round_, tenant, commit, sig, salt):\\n        key = (round_, tenant)\\n        if key in self.entries:\\n            return False  # one commitment per tenant per round\\n        self.entries[key] = (commit, sig, salt)\\n        return True\\n\\n    def challenge(self, round_, tenant, revealed_delta) -> dict:\\n        \\\"\\\"\\\"Dispute path: re-derive the commitment from the revealed update and re-verify the signature.\\\"\\\"\\\"\\n        commit, sig, salt = self.entries[(round_, tenant)]\\n        ok_hash = commitment(round_, tenant, revealed_delta, salt) == commit\\n        ok_sig = MLDSA.verify(self.alg, commit, sig, self.keys[tenant])\\n        return {\\\"reveal_matches_commitment\\\": ok_hash, \\\"signature_valid\\\": ok_sig,\\n                # a valid signature over a commitment that the revealed update does not match is\\n                # publicly checkable proof that this tenant equivocated\\n                \\\"provable_equivocation\\\": ok_sig and not ok_hash}\\n\\n    def slash(self, tenant, frac=0.5):\\n        self.bond[tenant] = max(0.0, self.bond[tenant] - frac)\\n        return self.bond[tenant]\\n\"}")
pathlib.Path('qbfl').mkdir(exist_ok=True)
for name, src in FILES.items():
    pathlib.Path('qbfl', name).write_text(src)
print('wrote', sorted(FILES))
print('\n✔ CELL COMPLETED: qbfl package written to disk')

In [ ]:
import sys, os, json, glob, time, platform, numpy as np, torch, pandas as pd
sys.path.insert(0, '.')
from qbfl import data as D, grid, analysis as A
from qbfl.fl import Config, run_one
pd.set_option('display.width', 230, 'display.max_rows', 400, 'display.max_columns', 40)
GPUS = [f'cuda:{i}' for i in range(torch.cuda.device_count())] or ['cpu']
os.makedirs('results', exist_ok=True)
import sklearn, scipy, pyarrow
VERS = dict(python=platform.python_version(), torch=torch.__version__, cuda=torch.version.cuda, numpy=np.__version__,
            pandas=pd.__version__, sklearn=sklearn.__version__, scipy=scipy.__version__, pyarrow=pyarrow.__version__,
            gpus=[torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])
try:
    import oqs; VERS['liboqs_python'] = getattr(oqs, '__version__', '?'); VERS['liboqs'] = oqs.oqs_version()
except Exception as e:
    VERS['liboqs'] = repr(e)
json.dump(VERS, open('results/versions.json', 'w'), indent=1); print(json.dumps(VERS, indent=1))
print('\n✔ CELL COMPLETED: libraries imported, versions recorded')

In [ ]:
tp, fp_ = D.split_pools(D.synthetic_pools(n_per=12000))
w_s = D.build_world(tp, n_tenants=12, n_train=500, n_test=1200, n_audit=2500)
for d in grid.FINAL_DEFENSES:
    for a in ('rule_aware', 'opt_adaptive_x4'):
        r = run_one(w_s, Config(defense=d, attack=a, mal_frac=0.25, rounds=4, eval_every=2, local_epochs=1, attack_start=2, dparams=grid.TUNE_GRIDS[d][0]), GPUS[0])
    print(f"{d:13s} ok")
print('SELF-TEST PASSED; scaler_fit =', w_s.meta['scaler_fit'])
print('\n✔ CELL COMPLETED: self-test passed')

In [ ]:
import kagglehub
SOURCES = list(D.KAGGLE_SOURCES)
paths = {k: D.download(k) for k in SOURCES}
print(paths)
print('\n✔ CELL COMPLETED: datasets downloaded')

In [ ]:
REPORT, raw = {}, {}
for k in SOURCES:
    t = time.time()
    X, y, fam, feat, rep = D.load_source(paths[k], max_rows=400_000, per_family_cap=60_000, seed=0, batch_rows=100_000, verbose=False)
    raw[k] = (X, y, fam, feat); REPORT[k] = rep
    print(f"{k}: {len(X):,} rows, dup removed {rep['dup_frac']:.1%}, {time.time()-t:.0f}s", rep['family_counts'])
common = [c for c in raw[SOURCES[0]][3] if all(c in raw[k][3] for k in SOURCES)]
pools = {k: (raw[k][0][:, [raw[k][3].index(c) for c in common]], raw[k][1], raw[k][2]) for k in SOURCES}
REPORT['common_features'] = common; D.report_json(REPORT, 'results/data_report.json')
assert all('dos' in set(pools[k][2]) for k in SOURCES)
print(len(common), 'shared features. DATA OK')
print('\n✔ CELL COMPLETED: data cleaned, deduplicated and aligned')

In [ ]:
SEEDS = json.loads('{"TUNE": [200, 201, 202], "MAIN": [300, 301, 302, 303, 304, 305, 306, 307, 308, 309], "MEM": [310, 311, 312, 313, 314, 315, 316, 317, 318, 319], "SETTLE": [320, 321, 322, 323, 324, 325, 326, 327, 328, 329, 330, 331, 332, 333, 334, 335, 336, 337, 338, 339, 340, 341, 342, 343, 344, 345, 346, 347, 348, 349], "STRONG": [350, 351, 352, 353, 354, 355, 356, 357, 358, 359], "LOSO": [360, 361, 362, 363, 364], "RECON": [365, 366, 367, 368, 369], "ACCT": [370, 371, 372, 373, 374, 375, 376, 377, 378, 379], "SENS": [380, 381, 382, 383, 384], "DETECT": [390, 391, 392, 393, 394], "KUBE": [400, 401, 402, 403, 404], "ENERGY": 500}')
SEEDS = {k: (tuple(v) if isinstance(v, list) else v) for k, v in SEEDS.items()}
FAILED = []
pools_tune, pools_final = D.split_pools(pools, seed=12345)
def world_from(p, **kw):
    a = dict(n_tenants=20, n_train=3000, n_test=6000, n_audit=9000, alpha=1.0, seed=0, scaler_fit='train'); a.update(kw)
    return D.build_world(p, **a)
world_tune = world_from(pools_tune, seed=1)
world_final = world_from(pools_final)
def go(world, cfgs, name):
    out = f'results/{name}'
    print(name, len(cfgs), 'configs on', GPUS, flush=True)
    if len(GPUS) > 1: grid.run_parallel(world, cfgs, out, GPUS, tag=name)
    else: grid.run_grid(world, cfgs, out + '.shard0.jsonl', GPUS[0], tag=name)
print('worlds built with scaler_fit =', world_final.meta['scaler_fit'], '|', len(world_final.tenants), 'tenants,', world_final.n_features, 'features')
print('\n✔ CELL COMPLETED: worlds built (scaler fitted on tenant training rows only)')

In [ ]:
BEST = json.loads('{"cosrep": {"lam": 0.8, "rho": 0.1}, "fedavg": {}, "median": {}, "mkrum": {"f_frac": 0.1}, "normgate": {"tau_mult": 1.25}, "probe_public": {"calib_q": 0.9, "delta_floor": 0.02}, "public_fam": {"kappa": 6.0, "s_min": 0.1, "use_rep": false}, "sealed": {"delta_floor": 0.1, "kappa": 3.0, "use_rep": false}, "sealed_abs": {"calib_q": 0.9, "delta_floor": 0.0}, "sealed_fam": {"kappa": 6.0, "s_min": 0.1, "use_rep": false}, "trmean": {"beta": 0.1}, "trustaware": {"lam": 0.7, "temp": 1.0}}')
json.dump(BEST, open('results/best_params.json', 'w'), indent=1, sort_keys=True)
print('settings selected by R1 (half A, seeds 200-202):', json.dumps(BEST, sort_keys=True))
print('\n✔ CELL COMPLETED: R1 settings loaded')

## H1B

In [ ]:
if RUN.get('H1B', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'H1B'), 'r3_h1b')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('H1B'); print('STAGE FAILED (continuing):', 'H1B', repr(e))
else:
    print('SKIPPED: H1B')
print('\n✔ CELL COMPLETED: H1B finished (or skipped)')

## ES

In [ ]:
if RUN.get('ES', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'ES'), 'r3_es')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('ES'); print('STAGE FAILED (continuing):', 'ES', repr(e))
else:
    print('SKIPPED: ES')
print('\n✔ CELL COMPLETED: ES finished (or skipped)')

## AGR

In [ ]:
if RUN.get('AGR', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'AGR'), 'r3_agr')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('AGR'); print('STAGE FAILED (continuing):', 'AGR', repr(e))
else:
    print('SKIPPED: AGR')
print('\n✔ CELL COMPLETED: AGR finished (or skipped)')

## FRAC

In [ ]:
if RUN.get('FRAC', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'FRAC'), 'r3_frac')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('FRAC'); print('STAGE FAILED (continuing):', 'FRAC', repr(e))
else:
    print('SKIPPED: FRAC')
print('\n✔ CELL COMPLETED: FRAC finished (or skipped)')

## KNOW

In [ ]:
if RUN.get('KNOW', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'KNOW'), 'r3_know')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('KNOW'); print('STAGE FAILED (continuing):', 'KNOW', repr(e))
else:
    print('SKIPPED: KNOW')
print('\n✔ CELL COMPLETED: KNOW finished (or skipped)')

## BUDGET2

In [ ]:
if RUN.get('BUDGET2', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'BUDGET2'), 'r3_budget2')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('BUDGET2'); print('STAGE FAILED (continuing):', 'BUDGET2', repr(e))
else:
    print('SKIPPED: BUDGET2')
print('\n✔ CELL COMPLETED: BUDGET2 finished (or skipped)')

## ARCH

In [ ]:
if RUN.get('ARCH', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'ARCH'), 'r3_arch')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('ARCH'); print('STAGE FAILED (continuing):', 'ARCH', repr(e))
else:
    print('SKIPPED: ARCH')
print('\n✔ CELL COMPLETED: ARCH finished (or skipped)')

## BUDGET8

In [ ]:
if RUN.get('BUDGET8', False):
    try:
        go(world_final, grid.r3_preset(BEST, 'BUDGET8'), 'r3_budget8')
    except Exception as e:
        import traceback; traceback.print_exc(); FAILED.append('BUDGET8'); print('STAGE FAILED (continuing):', 'BUDGET8', repr(e))
else:
    print('SKIPPED: BUDGET8')
print('\n✔ CELL COMPLETED: BUDGET8 finished (or skipped)')

## Pre-registered analysis (printed here; the paper's tables are regenerated locally by tools/make_tables.py)

In [ ]:
try:
    for f in sorted(glob.glob('results/r3_*.shard0.jsonl')):
        st = os.path.basename(f).split('.shard')[0]
        d = A.load(f'results/{st}*.jsonl'); print('\n==', st, len(d), 'runs')
        print(d.pivot_table(index='defense', columns=['attack', 'mal_frac'], values='final_rec_target', aggfunc='mean').round(3).to_string())
except Exception as e:
    import traceback; traceback.print_exc(); FAILED.append('ANALYSIS'); print('STAGE FAILED (continuing):', 'ANALYSIS', repr(e))
print('\n✔ CELL COMPLETED: pre-registered analysis printed')

In [ ]:
import shutil; shutil.make_archive('results_R3_bundle', 'zip', 'results')
print('files:', sorted(os.listdir('results')))
print('FAILED STAGES:', FAILED or 'none')
print('\n✔ CELL COMPLETED: results_R3_bundle.zip created')